# Práctica 6 · Aceleración en GPU con CUDA (Colab / Kaggle)

> **Cómputo Paralelo · 6BM1 · ESCOM-IPN · 27/1** · Unidad V · Práctica 6 · publicada el lunes 23 de noviembre de 2026 · Lab 1 martes 24 de noviembre 08:30 · Lab 2 martes 1 de diciembre 08:30 · entrega domingo 6 de diciembre 23:59

**Antes de ejecutar nada, activa la GPU:**

- **Colab:** *Entorno de ejecución → Cambiar tipo de entorno de ejecución → Acelerador de hardware: GPU (T4)* → Guardar.
- **Kaggle:** panel derecho *Settings → Accelerator → GPU T4 x2* (o P100). Kaggle pide verificación telefónica para usar GPU.

La primera celda debe mostrar una tarjeta NVIDIA. Si no la muestra, nada de lo que sigue corre: no sigas. Si Colab y Kaggle
niegan GPU, trabaja la correctitud en tu laptop con `NUMBA_ENABLE_CUDASIM=1` (ver README, "Sin GPU") y reintenta después.

Protocolo del curso: **anota la GPU asignada y la fecha** (cambian entre sesiones); cada programa hace 1 calentamiento +
5 repeticiones y reporta la **mediana**; las copias anfitrión↔dispositivo se reportan **aparte** y también sumadas.
Los órdenes de magnitud del README son para una T4; los tuyos dependen de la GPU que te toque.

Sigue el `README.md` de la práctica: **predice por escrito antes de ejecutar cada celda.**

In [ ]:
!nvidia-smi
!nvcc --version | tail -2
!lscpu | grep -E 'Model name|^CPU\(s\)|Thread|MHz' ; free -h | head -2

In [ ]:
# Arquitectura para nvcc: T4 (Colab) -> sm_75; P100 (Kaggle) -> sm_60. Se detecta; si falla, queda sm_75.
import subprocess
ARCH = "sm_75"
try:
    cap = subprocess.check_output(["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"], text=True).strip().splitlines()[0]
    ARCH = "sm_" + cap.replace(".", "")
except Exception as exc:  # sin GPU o nvidia-smi viejo
    print("No se pudo detectar la GPU:", exc)
print("ARCH =", ARCH)

## 1 · Escribir los fuentes en el disco de la sesión

Cada celda `%%writefile` guarda un archivo. Son los mismos de `src/` (y `graficar.py`) del repositorio. Si editas uno aquí (por ejemplo, al corregir los `TODO` de la Parte B), vuelve a ejecutar su celda y recompila (sección 2).

In [ ]:
%%writefile common.cuh
// common.cuh — utilidades de anfitrión compartidas por los programas CUDA de la Práctica 6.
// Nada de aquí corre en la GPU. Contiene: revisión de errores (CUDA_CHECK), reloj de CPU,
// mediana, memoria del anfitrión (paginable o fija), información del dispositivo, ocupación,
// el ESQUEMA DEL CSV del curso (el mismo de la sesión S04) y los generadores de datos de la
// Parte C, que son idénticos a los de las referencias en C (c_histograma_cpu.c, c_jacobi_cpu.c)
// para que los resultados sean comparables entre programas.
#pragma once
#include <cstdio>
#include <cstdlib>
#include <cstring>
#include <cmath>
#include <ctime>
#include <cuda_runtime.h>

// Toda llamada a la API de CUDA se envuelve en CUDA_CHECK. Si falla, imprime archivo, línea y
// mensaje, y termina. Sin esto un error pasa en silencio y el programa "parece" funcionar.
#define CUDA_CHECK(call)                                                          \
    do {                                                                          \
        cudaError_t err__ = (call);                                               \
        if (err__ != cudaSuccess) {                                               \
            fprintf(stderr, "Error CUDA en %s:%d: %s\n", __FILE__, __LINE__,      \
                    cudaGetErrorString(err__));                                   \
            exit(EXIT_FAILURE);                                                   \
        }                                                                         \
    } while (0)

// Protocolo de medición del curso: 1 calentamiento (no se cuenta) + REPS repeticiones, mediana.
#define REPS 5

// Tiempos de GPU de una configuración: medianas de REPS repeticiones, en milisegundos.
typedef struct {
    double h2d_ms;     // copias anfitrión -> dispositivo
    double kernel_ms;  // solo el (los) kernel(s)
    double d2h_ms;     // copia dispositivo -> anfitrión
} GpuTimes;

static inline double cpu_seconds(void) {
    struct timespec ts;
    clock_gettime(CLOCK_MONOTONIC, &ts);
    return (double)ts.tv_sec + 1e-9 * (double)ts.tv_nsec;
}

static inline int compare_double(const void *a, const void *b) {
    double da = *(const double *)a, db = *(const double *)b;
    return (da > db) - (da < db);
}

// Ordena v en su lugar y regresa la mediana (REPS es impar). Tras llamarla, v[0] es el mínimo
// y v[k-1] el máximo.
static inline double median_of(double *v, int k) {
    qsort(v, (size_t)k, sizeof(double), compare_double);
    return v[k / 2];
}

// Memoria del anfitrión: paginable (malloc) o fija (cudaMallocHost). La fija permite copias
// por DMA sin paso intermedio y copias asíncronas de verdad; se usa en la extensión opcional.
static inline void *host_alloc(size_t bytes, int pinned) {
    void *p = NULL;
    if (pinned) {
        CUDA_CHECK(cudaMallocHost(&p, bytes));
    } else {
        p = malloc(bytes);
        if (!p) {
            fprintf(stderr, "Sin memoria en el anfitrión (%zu bytes)\n", bytes);
            exit(EXIT_FAILURE);
        }
    }
    return p;
}

static inline void host_free(void *p, int pinned) {
    if (pinned) CUDA_CHECK(cudaFreeHost(p));
    else free(p);
}

// Nombre de la GPU sin comas (va dentro de un CSV) y resumen del dispositivo (protocolo §1.4).
static inline void describe_device(cudaDeviceProp *prop, char *name_out, size_t cap) {
    int dev = 0;
    CUDA_CHECK(cudaGetDevice(&dev));
    CUDA_CHECK(cudaGetDeviceProperties(prop, dev));
    snprintf(name_out, cap, "%s", prop->name);
    for (char *p = name_out; *p; ++p)
        if (*p == ',') *p = ' ';
    int mem_clock_khz = 0, bus_width_bits = 0;
    CUDA_CHECK(cudaDeviceGetAttribute(&mem_clock_khz, cudaDevAttrMemoryClockRate, dev));
    CUDA_CHECK(cudaDeviceGetAttribute(&bus_width_bits, cudaDevAttrGlobalMemoryBusWidth, dev));
    double peak_gbs = 2.0 * mem_clock_khz * 1e3 * (bus_width_bits / 8.0) / 1e9;  // DDR: 2 por ciclo
    printf("GPU: %s · %d SM · %d hilos/SM · cc %d.%d · ancho de banda teórico %.0f GB/s\n", name_out,
           prop->multiProcessorCount, prop->maxThreadsPerMultiProcessor, prop->major, prop->minor,
           peak_gbs);
}

// Ocupación teórica (%) de un kernel con un tamaño de bloque: warps residentes por SM que el
// kernel permite, entre el máximo de warps del SM. Es plantilla porque recibe el kernel mismo.
template <typename KernelT>
static inline double occupancy_pct(KernelT kernel, int block, size_t dyn_smem,
                                   const cudaDeviceProp *prop) {
    int blocks_per_sm = 0;
    CUDA_CHECK(cudaOccupancyMaxActiveBlocksPerMultiprocessor(&blocks_per_sm, kernel, block, dyn_smem));
    int warps_per_block = (block + 31) / 32;  // un bloque de 48 hilos ocupa 2 warps, uno a medias
    int max_warps = prop->maxThreadsPerMultiProcessor / 32;
    return 100.0 * blocks_per_sm * warps_per_block / max_warps;
}

// ---------------------------------------------------------------------------------------------
// Esquema del CSV del curso (sesión S04). TODO programa de la práctica (Partes A, B y C) agrega
// filas con este encabezado; graficar.py y resultados/README.md lo documentan.
//   t_* son medianas de REPS repeticiones en ms; t_total = t_h2d + t_kernel + t_d2h;
//   speedup_kernel = t_cpu / t_kernel; speedup_total = t_cpu / t_total;
//   ancho_banda_gbs = bytes útiles movidos por el kernel (entradas + salidas) / t_kernel.
// ---------------------------------------------------------------------------------------------
static const char *CSV_HEADER =
    "experimento,kernel,n,block,grid,ocupacion_pct,memoria,t_h2d_ms,t_kernel_ms,t_d2h_ms,t_total_ms,"
    "t_cpu_ms,speedup_kernel,speedup_total,ancho_banda_gbs,origen,dispositivo\n";

// Abre el CSV en modo "agregar" y escribe el encabezado solo si el archivo está vacío.
static inline FILE *open_csv(const char *path) {
    if (!path) return NULL;
    FILE *f = fopen(path, "a");
    if (!f) {
        fprintf(stderr, "No se pudo abrir %s\n", path);
        exit(EXIT_FAILURE);
    }
    fseek(f, 0, SEEK_END);
    if (ftell(f) == 0) fputs(CSV_HEADER, f);
    return f;
}

static inline void print_table_header(void) {
    printf("%-18s %10s %6s %8s %6s %9s %10s %9s %10s %10s %8s %8s %7s  %s\n", "kernel", "n", "bloque",
           "malla", "ocup%", "H2D ms", "kernel ms", "D2H ms", "total ms", "CPU ms", "S_kern", "S_total",
           "GB/s", "verif");
}

// Imprime una fila de la tabla en pantalla y, si csv != NULL, la fila equivalente del CSV.
static inline void emit_row(FILE *csv, const char *experiment, const char *kernel_name, long n, int block,
                            long grid, double occ, const char *memory, GpuTimes t, double cpu_ms,
                            double useful_bytes, int ok, const char *device) {
    double total = t.h2d_ms + t.kernel_ms + t.d2h_ms;
    double gbs = useful_bytes / (t.kernel_ms * 1e-3) / 1e9;
    double s_kernel = cpu_ms / t.kernel_ms, s_total = cpu_ms / total;
    printf("%-18s %10ld %6d %8ld %6.0f %9.4f %10.4f %9.4f %10.4f %10.4f %8.2f %8.2f %7.1f  %s\n",
           kernel_name, n, block, grid, occ, t.h2d_ms, t.kernel_ms, t.d2h_ms, total, cpu_ms, s_kernel,
           s_total, gbs, ok ? "OK" : "FALLA");
    if (csv)
        fprintf(csv, "%s,%s,%ld,%d,%ld,%.1f,%s,%.6f,%.6f,%.6f,%.6f,%.6f,%.4f,%.4f,%.2f,gpu,%s\n",
                experiment, kernel_name, n, block, grid, occ, memory, t.h2d_ms, t.kernel_ms, t.d2h_ms,
                total, cpu_ms, s_kernel, s_total, gbs, device);
}

// ---------------------------------------------------------------------------------------------
// Generadores de datos de la Parte C. Son EXACTAMENTE los mismos que usan c_histograma_cpu.c y
// c_jacobi_cpu.c: tu kernel debe consumir estos datos para que la comparación tenga sentido.
// ---------------------------------------------------------------------------------------------

// Opción 1 · histograma/reducción: n bytes pseudoaleatorios (hash multiplicativo de Knuth).
static inline void init_histogram_data(unsigned char *data, long n) {
    for (long i = 0; i < n; ++i) {
        unsigned int x = (unsigned int)i * 2654435761u;  // la multiplicación desborda a propósito (mod 2^32)
        data[i] = (unsigned char)((x >> 24) & 0xFFu);
    }
}

// Opción 2 · Jacobi 2D: malla alto×ancho aplanada por filas; la fila 0 vale 100, el resto 0.
// La frontera (fila 0, última fila, columnas 0 y ancho-1) no cambia durante la iteración.
static inline void init_jacobi_grid(float *grid, int height, int width) {
    for (int r = 0; r < height; ++r)
        for (int c = 0; c < width; ++c) grid[(size_t)r * width + c] = (r == 0) ? 100.0f : 0.0f;
}

In [ ]:
%%writefile a_vectores.cu
// a_vectores.cu — Práctica 6 · Parte A: suma de vectores (c = a + b) y SAXPY (y = α·x + y) en CUDA C,
// con medición por separado de las copias H2D, el kernel y la copia D2H usando cudaEvent_t.
//
// Compilar (Colab/Kaggle con GPU):   nvcc -O2 -arch=sm_75 a_vectores.cu -o a_vectores     (sm_60 en P100)
// Verificar que compila sin GPU:     nvcc -c -arch=sm_75 a_vectores.cu -o a_vectores.o
//
// Uso:
//   ./a_vectores uno    <n> <block> [salida.csv] [pinned]   una configuración, los dos kernels (Lab 1)
//   ./a_vectores umbral [block=256] [salida.csv] [pinned]   n = 2^10, 2^12, ..., 2^26 con bloque fijo
//   ./a_vectores bloque [n=2^24]    [salida.csv] [pinned]   bloque = 32, 64, ..., 1024 con n fijo (Parte D)
//
// Cada configuración: 1 calentamiento + 5 repeticiones; se reporta la mediana de H2D, kernel y D2H.
// T(1) es la versión secuencial en C compilada en este mismo programa (mediana de 5 también).
// Todo se verifica contra la CPU: "sin error de CUDA" no significa "correcto" (idea errónea 27).
// Las filas se AGREGAN al CSV con el esquema del curso (ver common.cuh); varias corridas se juntan.
#include "common.cuh"

static const float ALPHA = 2.0f;
static const int BLOCK_SIZES[] = {32, 64, 128, 256, 512, 1024};
static const int N_BLOCK_SIZES = (int)(sizeof(BLOCK_SIZES) / sizeof(BLOCK_SIZES[0]));
static const int LOG2_N_MIN = 10, LOG2_N_MAX = 26, LOG2_N_STEP = 2;

typedef enum { K_VECTOR_ADD = 0, K_SAXPY = 1 } KernelId;
static const char *KERNEL_NAMES[] = {"vector_add", "saxpy"};

// ---------- Código de dispositivo: un hilo por elemento ----------

// Cada hilo calcula UN elemento. El índice global se arma con la posición del bloque en la malla,
// el tamaño del bloque y la posición del hilo dentro del bloque.
__global__ void vector_add_kernel(const float *a, const float *b, float *c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) c[i] = a[i] + b[i];  // guardia: los hilos sobrantes del último bloque no tocan memoria
}

// SAXPY: 2 FLOP por 12 bytes movidos (intensidad aritmética ≈ 0.17 FLOP/byte: memory-bound).
__global__ void saxpy_kernel(float alpha, const float *x, const float *y, float *out, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) out[i] = alpha * x[i] + y[i];
}

// ---------- Código de anfitrión ----------

// Referencias secuenciales: el "mejor programa secuencial" contra el que se calcula el speedup.
static void vector_add_cpu(const float *a, const float *b, float *c, int n) {
    for (int i = 0; i < n; ++i) c[i] = a[i] + b[i];
}

static void saxpy_cpu(float alpha, const float *x, const float *y, float *out, int n) {
    for (int i = 0; i < n; ++i) out[i] = alpha * x[i] + y[i];
}

// Misma inicialización que a_vectores_cpu.c y a_vectores_numba.py: enteros pequeños, por lo que la
// aritmética en float es exacta y el checksum es reproducible entre los tres programas.
static void init_inputs(float *a, float *b, int n) {
    for (int i = 0; i < n; ++i) {
        a[i] = (float)(i % 7);
        b[i] = (float)(i % 13);
    }
}

static double max_abs_error(const float *got, const float *ref, int n, double *checksum) {
    double max_err = 0.0, sum = 0.0;
    for (int i = 0; i < n; ++i) {
        double err = fabs((double)got[i] - (double)ref[i]);
        if (err > max_err) max_err = err;
        sum += got[i];
    }
    *checksum = sum;
    return max_err;
}

// Mediana del tiempo secuencial (1 calentamiento + REPS). Deja el resultado de referencia en c.
static double measure_cpu_ms(KernelId k, int n, const float *a, const float *b, float *c) {
    double t[REPS];
    for (int r = -1; r < REPS; ++r) {
        double t0 = cpu_seconds();
        if (k == K_VECTOR_ADD) vector_add_cpu(a, b, c, n);
        else saxpy_cpu(ALPHA, a, b, c, n);
        double dt = (cpu_seconds() - t0) * 1e3;
        if (r >= 0) t[r] = dt;
    }
    return median_of(t, REPS);
}

// Mide H2D, kernel y D2H por separado con cuatro eventos en el stream 0. Un evento es una marca
// que la GPU "completa" cuando llega a ella; cudaEventElapsedTime da los ms entre dos marcas.
// Deja la salida de la última repetición en h_c para verificarla.
static GpuTimes measure_gpu(KernelId k, int n, int block, const float *h_a, const float *h_b, float *h_c) {
    size_t bytes = (size_t)n * sizeof(float);
    float *d_a = NULL, *d_b = NULL, *d_c = NULL;
    CUDA_CHECK(cudaMalloc((void **)&d_a, bytes));
    CUDA_CHECK(cudaMalloc((void **)&d_b, bytes));
    CUDA_CHECK(cudaMalloc((void **)&d_c, bytes));

    cudaEvent_t ev[4];
    for (int i = 0; i < 4; ++i) CUDA_CHECK(cudaEventCreate(&ev[i]));

    int grid = (n + block - 1) / block;  // redondeo hacia arriba: cubre todo n
    double h2d[REPS], ker[REPS], d2h[REPS];

    for (int r = -1; r < REPS; ++r) {  // r = -1 es el calentamiento (no se cuenta)
        CUDA_CHECK(cudaEventRecord(ev[0], 0));
        CUDA_CHECK(cudaMemcpy(d_a, h_a, bytes, cudaMemcpyHostToDevice));
        CUDA_CHECK(cudaMemcpy(d_b, h_b, bytes, cudaMemcpyHostToDevice));
        CUDA_CHECK(cudaEventRecord(ev[1], 0));
        if (k == K_VECTOR_ADD) vector_add_kernel<<<grid, block>>>(d_a, d_b, d_c, n);
        else saxpy_kernel<<<grid, block>>>(ALPHA, d_a, d_b, d_c, n);
        CUDA_CHECK(cudaEventRecord(ev[2], 0));
        CUDA_CHECK(cudaMemcpy(h_c, d_c, bytes, cudaMemcpyDeviceToHost));
        CUDA_CHECK(cudaEventRecord(ev[3], 0));
        CUDA_CHECK(cudaEventSynchronize(ev[3]));  // sin esto se mide el encolado, no el trabajo
        CUDA_CHECK(cudaGetLastError());           // ¿falló el lanzamiento? (p. ej. bloque > 1024)
        if (r >= 0) {
            float ms;
            CUDA_CHECK(cudaEventElapsedTime(&ms, ev[0], ev[1])); h2d[r] = ms;
            CUDA_CHECK(cudaEventElapsedTime(&ms, ev[1], ev[2])); ker[r] = ms;
            CUDA_CHECK(cudaEventElapsedTime(&ms, ev[2], ev[3])); d2h[r] = ms;
        }
    }

    GpuTimes t;
    t.h2d_ms = median_of(h2d, REPS);
    t.kernel_ms = median_of(ker, REPS);
    t.d2h_ms = median_of(d2h, REPS);

    for (int i = 0; i < 4; ++i) CUDA_CHECK(cudaEventDestroy(ev[i]));
    CUDA_CHECK(cudaFree(d_a));
    CUDA_CHECK(cudaFree(d_b));
    CUDA_CHECK(cudaFree(d_c));
    return t;
}

static double occupancy_of(KernelId k, int block, const cudaDeviceProp *prop) {
    return (k == K_VECTOR_ADD) ? occupancy_pct(vector_add_kernel, block, 0, prop)
                               : occupancy_pct(saxpy_kernel, block, 0, prop);
}

// Una configuración (n, block) para los dos kernels: mide CPU, mide GPU, verifica, imprime y guarda.
// Regresa el número de verificaciones que fallaron.
static int run_config(const char *experiment, int n, int block, int pinned, FILE *csv,
                      const cudaDeviceProp *prop, const char *device, int verbose_checksum) {
    size_t bytes = (size_t)n * sizeof(float);
    float *h_a = (float *)host_alloc(bytes, pinned), *h_b = (float *)host_alloc(bytes, pinned);
    float *h_c = (float *)host_alloc(bytes, pinned), *h_ref = (float *)malloc(bytes);
    if (!h_ref) exit(EXIT_FAILURE);
    init_inputs(h_a, h_b, n);
    int failures = 0;
    for (int k = 0; k < 2; ++k) {
        double cpu_ms = measure_cpu_ms((KernelId)k, n, h_a, h_b, h_ref);
        GpuTimes t = measure_gpu((KernelId)k, n, block, h_a, h_b, h_c);
        double checksum = 0.0;
        double err = max_abs_error(h_c, h_ref, n, &checksum);
        int ok = (err <= 1e-5);
        if (!ok) ++failures;
        long grid = (n + block - 1) / block;
        emit_row(csv, experiment, KERNEL_NAMES[k], n, block, grid, occupancy_of((KernelId)k, block, prop),
                 pinned ? "pinned" : "pageable", t, cpu_ms, 3.0 * (double)bytes, ok, device);
        if (verbose_checksum)
            printf("    %s: checksum=%.1f max_abs_err=%.3g (compara con a_vectores_cpu %d y el espejo Numba)\n",
                   KERNEL_NAMES[k], checksum, err, n);
    }
    host_free(h_a, pinned);
    host_free(h_b, pinned);
    host_free(h_c, pinned);
    free(h_ref);
    return failures;
}

static void usage(const char *prog) {
    fprintf(stderr,
            "Uso:\n"
            "  %s uno    <n> <block> [salida.csv] [pinned]\n"
            "  %s umbral [block=256] [salida.csv] [pinned]\n"
            "  %s bloque [n=16777216] [salida.csv] [pinned]\n",
            prog, prog, prog);
    exit(EXIT_FAILURE);
}

int main(int argc, char **argv) {
    if (argc < 2) usage(argv[0]);
    const char *mode = argv[1];
    // "pinned" puede ir en cualquier posición; lo demás son argumentos posicionales.
    int pinned = 0;
    const char *pos[4] = {NULL, NULL, NULL, NULL};
    int n_pos = 0;
    for (int i = 2; i < argc; ++i) {
        if (strcmp(argv[i], "pinned") == 0) pinned = 1;
        else if (n_pos < 4) pos[n_pos++] = argv[i];
    }

    cudaDeviceProp prop;
    char device[256];
    describe_device(&prop, device, sizeof device);
    printf("memoria del anfitrión: %s · protocolo: 1 calentamiento + %d repeticiones, mediana\n",
           pinned ? "pinned" : "pageable", REPS);

    int failures = 0;
    if (strcmp(mode, "uno") == 0) {
        if (n_pos < 2) usage(argv[0]);
        int n = atoi(pos[0]), block = atoi(pos[1]);
        if (n <= 0 || block <= 0) usage(argv[0]);
        // No se limita block a 1024 a propósito: con 2048 el lanzamiento falla y CUDA_CHECK lo reporta.
        FILE *csv = open_csv(pos[2]);
        printf("Experimento 'uno': n = %d (%.2f MB por vector), bloque = %d\n", n,
               (double)n * sizeof(float) / 1048576.0, block);
        print_table_header();
        failures += run_config("uno", n, block, pinned, csv, &prop, device, 1);
        if (csv) fclose(csv);
    } else if (strcmp(mode, "umbral") == 0) {
        int block = pos[0] ? atoi(pos[0]) : 256;
        if (block < 1 || block > 1024) usage(argv[0]);
        if (block % 32 != 0)
            printf("Aviso: %d no es múltiplo de 32; el último warp de cada bloque va incompleto\n", block);
        FILE *csv = open_csv(pos[1]);
        printf("Experimento 'umbral': bloque = %d, n = 2^%d .. 2^%d\n", block, LOG2_N_MIN, LOG2_N_MAX);
        print_table_header();
        for (int log2n = LOG2_N_MIN; log2n <= LOG2_N_MAX; log2n += LOG2_N_STEP)
            failures += run_config("umbral", 1 << log2n, block, pinned, csv, &prop, device, 0);
        if (csv) fclose(csv);
    } else if (strcmp(mode, "bloque") == 0) {
        int n = pos[0] ? atoi(pos[0]) : (1 << 24);
        if (n <= 0 || n > (1 << 27)) usage(argv[0]);
        FILE *csv = open_csv(pos[1]);
        printf("Experimento 'bloque': n = %d (%.1f MB por vector), bloques de %d a %d\n", n,
               (double)n * sizeof(float) / 1048576.0, BLOCK_SIZES[0], BLOCK_SIZES[N_BLOCK_SIZES - 1]);
        print_table_header();
        for (int i = 0; i < N_BLOCK_SIZES; ++i)
            failures += run_config("bloque", n, BLOCK_SIZES[i], pinned, csv, &prop, device, 0);
        if (csv) fclose(csv);
    } else {
        usage(argv[0]);
    }

    printf("Verificación contra la CPU: %s%s\n", failures ? "FALLA" : "OK",
           pos[2] || (strcmp(mode, "uno") != 0 && pos[1]) ? " · filas agregadas al CSV" : "");
    CUDA_CHECK(cudaDeviceReset());
    return failures ? 2 : 0;
}

In [ ]:
%%writefile a_vectores_cpu.c
/*
 * a_vectores_cpu.c — Referencia secuencial de la Parte A: suma de vectores y SAXPY.
 *
 * Uso:      ./a_vectores_cpu [n=16777216] [vector_add|saxpy|ambos]
 * Compilar: gcc -std=c11 -O2 -Wall -Wextra a_vectores_cpu.c -o a_vectores_cpu
 *
 * Misma inicialización y misma aritmética que a_vectores.cu y a_vectores_numba.py, por lo que el
 * checksum coincide entre los tres programas para el mismo n. Es el T(1) del protocolo cuando la
 * CPU de Colab/Kaggle es la que se compara (el .cu ya lo mide internamente; este programa sirve
 * para verificar sin GPU y para medir T(1) en tu laptop).
 * Protocolo: 1 calentamiento + 5 repeticiones, mediana; se reportan también mínimo y máximo.
 */
#define _POSIX_C_SOURCE 200809L
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <math.h>
#include <time.h>

#define REPS 5
static const float ALPHA = 2.0f;

static double now_s(void)
{
    struct timespec ts;
    clock_gettime(CLOCK_MONOTONIC, &ts);
    return (double)ts.tv_sec + (double)ts.tv_nsec * 1e-9;
}

/* Los ciclos que la GPU reparte entre miles de hilos: aquí los hace un solo núcleo. */
static void vector_add_cpu(const float *a, const float *b, float *c, int n)
{
    for (int i = 0; i < n; i++)
        c[i] = a[i] + b[i];
}

static void saxpy_cpu(float alpha, const float *x, const float *y, float *out, int n)
{
    for (int i = 0; i < n; i++)
        out[i] = alpha * x[i] + y[i];
}

static int cmp_double(const void *x, const void *y)
{
    double dx = *(const double *)x, dy = *(const double *)y;
    return (dx > dy) - (dx < dy);
}

/* Corre un kernel de CPU con el protocolo y verifica contra la fórmula cerrada. Regresa 1 si OK. */
static int run_kernel(int which, const float *a, const float *b, float *c, int n)
{
    const char *name = which == 0 ? "vector_add" : "saxpy";
    double t[REPS];
    for (int r = -1; r < REPS; r++) {
        double t0 = now_s();
        if (which == 0)
            vector_add_cpu(a, b, c, n);
        else
            saxpy_cpu(ALPHA, a, b, c, n);
        double dt = now_s() - t0;
        if (r >= 0)
            t[r] = dt;
    }
    qsort(t, REPS, sizeof(double), cmp_double);
    double t_med = t[REPS / 2];

    double checksum = 0.0, max_abs_err = 0.0;
    for (int i = 0; i < n; i++) {
        double expected = (which == 0) ? (double)(i % 7) + (double)(i % 13)
                                       : (double)ALPHA * (double)(i % 7) + (double)(i % 13);
        double diff = fabs((double)c[i] - expected);
        if (diff > max_abs_err)
            max_abs_err = diff;
        checksum += c[i];
    }
    int ok = (max_abs_err <= 1e-5);
    double bw = (3.0 * (double)n * sizeof(float) / 1e9) / (t_med > 0 ? t_med : 1e-12);
    printf("kernel=%s n=%d t_cpu_ms=%.4f (min %.4f, max %.4f) ancho_banda_GB_s=%.1f "
           "checksum=%.1f max_abs_err=%.3g verificacion=%s\n",
           name, n, t_med * 1e3, t[0] * 1e3, t[REPS - 1] * 1e3, bw, checksum, max_abs_err,
           ok ? "OK" : "FALLA");
    return ok;
}

int main(int argc, char **argv)
{
    int n = (argc > 1) ? atoi(argv[1]) : (1 << 24);
    const char *which = (argc > 2) ? argv[2] : "ambos";
    if (n <= 0) {
        fprintf(stderr, "uso: %s [n>0] [vector_add|saxpy|ambos]\n", argv[0]);
        return 1;
    }
    size_t bytes = (size_t)n * sizeof(float);
    float *a = malloc(bytes), *b = malloc(bytes), *c = malloc(bytes);
    if (!a || !b || !c) {
        fprintf(stderr, "Sin memoria para n=%d\n", n);
        return 1;
    }
    /* Misma inicialización que las versiones CUDA y Numba. */
    for (int i = 0; i < n; i++) {
        a[i] = (float)(i % 7);
        b[i] = (float)(i % 13);
    }
    memset(c, 0, bytes);

    printf("a_vectores_cpu: n=%d reps=%d (1 calentamiento + %d, mediana)\n", n, REPS, REPS);
    int ok = 1;
    if (strcmp(which, "saxpy") != 0)
        ok &= run_kernel(0, a, b, c, n);
    if (strcmp(which, "vector_add") != 0)
        ok &= run_kernel(1, a, b, c, n);

    free(a);
    free(b);
    free(c);
    return ok ? 0 : 2;
}

In [ ]:
%%writefile a_vectores_numba.py
#!/usr/bin/env python3
"""a_vectores_numba.py — Espejo en Numba CUDA de los kernels de a_vectores.cu (Parte A).

Uso:
    python3 a_vectores_numba.py [n] [block] [vector_add|saxpy|ambos]
Sin GPU (simulador en CPU: verifica la LÓGICA, los tiempos no significan nada; usa n pequeño):
    NUMBA_ENABLE_CUDASIM=1 python3 a_vectores_numba.py 1000 256

Mismo índice global, misma guardia, misma inicialización y mismo checksum que a_vectores.cu y
a_vectores_cpu.c. Con GPU real mide el kernel con eventos de CUDA (cuda.event) y el total con
copias usando perf_counter + cuda.synchronize(); 1 calentamiento (incluye la compilación JIT) + 5.
Bajo CUDASIM se hace una sola repetición: el tiempo del simulador no significa nada y cada hilo
se ejecuta en Python (n = 1000 tarda ~1 s; n = 20 000, ~20 s).
Termina con código 0 si todas las verificaciones contra NumPy pasan.
"""
import os
import sys
import time

import numpy as np
from numba import cuda

CUDASIM = os.environ.get("NUMBA_ENABLE_CUDASIM", "0") == "1"
REPS = 5
ALPHA = np.float32(2.0)


@cuda.jit
def vector_add_kernel(a, b, c, n):
    # Índice global igual que en CUDA C (cuda.grid(1) es la abreviatura equivalente).
    i = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
    if i < n:  # guardia
        c[i] = a[i] + b[i]


@cuda.jit
def saxpy_kernel(alpha, x, y, out, n):
    i = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
    if i < n:
        out[i] = alpha * x[i] + y[i]


def median(values):
    return sorted(values)[len(values) // 2]


def run(name, n, block):
    grid = (n + block - 1) // block  # división entera redondeada hacia arriba
    # Anfitrión: misma inicialización que a_vectores.cu / a_vectores_cpu.c
    a = (np.arange(n) % 7).astype(np.float32)
    b = (np.arange(n) % 13).astype(np.float32)
    c = np.zeros(n, dtype=np.float32)
    ref = (a + b) if name == "vector_add" else (ALPHA * a + b)

    def launch(d_a, d_b, d_c):
        if name == "vector_add":
            vector_add_kernel[grid, block](d_a, d_b, d_c, n)
        else:
            saxpy_kernel[grid, block](ALPHA, d_a, d_b, d_c, n)

    # Calentamiento (en GPU real incluye la compilación JIT del kernel y la creación del contexto)
    d_a, d_b, d_c = cuda.to_device(a), cuda.to_device(b), cuda.to_device(c)
    launch(d_a, d_b, d_c)
    cuda.synchronize()

    kernel_ms, total_ms = [], []
    reps = 1 if CUDASIM else REPS  # en el simulador el tiempo no es una medición: una repetición basta
    for _ in range(reps):
        # --- total con copias: H2D + kernel + D2H, reloj de CPU con sincronización ---
        t0 = time.perf_counter()
        d_a, d_b = cuda.to_device(a), cuda.to_device(b)       # equivalente de cudaMemcpy H2D
        launch(d_a, d_b, d_c)
        d_c.copy_to_host(c)                                   # D2H; sincroniza al terminar
        cuda.synchronize()
        total_ms.append((time.perf_counter() - t0) * 1e3)
        # --- solo kernel: eventos de CUDA (en CUDASIM se emulan con el reloj de CPU) ---
        e0, e1 = cuda.event(), cuda.event()
        e0.record()
        launch(d_a, d_b, d_c)
        e1.record()
        e1.synchronize()  # el lanzamiento es asíncrono: sin esto se mide casi nada
        kernel_ms.append(e0.elapsed_time(e1))  # ms entre las dos marcas, en el reloj de la GPU

    d_c.copy_to_host(c)
    max_abs_err = float(np.max(np.abs(c - ref)))
    checksum = float(np.sum(c, dtype=np.float64))
    ok = max_abs_err <= 1e-5
    print(f"kernel={name} n={n} block={block} grid={grid} hilos_lanzados={grid * block} "
          f"modo={'CUDASIM' if CUDASIM else 'GPU'}")
    print(f"    t_kernel_ms={median(kernel_ms):.4f} t_total_ms={median(total_ms):.4f} "
          f"checksum={checksum:.1f} max_abs_err={max_abs_err:.3g} verificacion={'OK' if ok else 'FALLA'}")
    return ok


def main():
    default_n = 2048 if CUDASIM else 1 << 24  # bajo CUDASIM cada hilo se simula en Python
    n = int(sys.argv[1]) if len(sys.argv) > 1 else default_n
    block = int(sys.argv[2]) if len(sys.argv) > 2 else 256
    which = sys.argv[3] if len(sys.argv) > 3 else "ambos"
    if n <= 0 or block <= 0 or block > 1024 or which not in ("vector_add", "saxpy", "ambos"):
        print("uso: a_vectores_numba.py [n>0] [block 1..1024] [vector_add|saxpy|ambos]")
        sys.exit(1)
    if CUDASIM and n > 20000:
        print("aviso: CUDASIM con n > 20000 puede tardar varios minutos", file=sys.stderr)
    ok = True
    if which != "saxpy":
        ok &= run("vector_add", n, block)
    if which != "vector_add":
        ok &= run("saxpy", n, block)
    sys.exit(0 if ok else 2)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile a_matmul_tiled.cu
// a_matmul_tiled.cu — Práctica 6 · Parte A (código COMPLETO para leer, predecir, ejecutar y medir):
// multiplicación de matrices C = A·B (n×n, float, guardadas por filas) en dos versiones.
//   matmul_naive  un hilo por elemento de C; lee una fila de A y una columna de B de memoria GLOBAL
//                 (2n lecturas globales por hilo).
//   matmul_tiled  por mosaicos (tiling): cada bloque de TILE×TILE hilos recorre la dimensión k en fases;
//                 en cada fase copia un mosaico de A y uno de B a memoria COMPARTIDA (__shared__),
//                 sincroniza, y todos los hilos del bloque reutilizan esos datos TILE veces
//                 (2·⌈n/TILE⌉ lecturas globales por hilo).
// Es el mismo kernel que rastreaste en la práctica guiada de la sesión S02 (martes 24 de noviembre), ahora
// con la medición del curso: H2D, kernel y D2H por separado con cudaEvent_t y filas en el CSV.
//
// Compilar (GPU):     nvcc -O2 -arch=sm_75 a_matmul_tiled.cu -o a_matmul_tiled        (-DTILE=8|16|32)
// Verificar sin GPU:  nvcc -c -arch=sm_75 a_matmul_tiled.cu -o a_matmul_tiled.o
// Uso:                ./a_matmul_tiled [n=512] [salida.csv]
//
// Verificación contra la CPU con tolerancia (nunca con ==: la GPU usa FMA y el orden de suma cambia el
// redondeo). Medición: 1 calentamiento + 5 repeticiones de H2D + kernel + D2H, mediana de cada tramo.
// En el CSV, block = TILE·TILE (hilos del bloque), grid = bloques totales y ancho_banda_gbs usa los
// 12·n² bytes útiles (A, B y C una vez); la tabla imprime además GFLOP/s.
#include "common.cuh"

#ifndef TILE
#define TILE 16
#endif

// ---------- Kernel ingenuo: 2n lecturas de memoria global por hilo ----------
__global__ void matmul_naive(const float *A, const float *B, float *C, int n) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    if (row < n && col < n) {  // aquí la guardia puede envolver todo: este kernel no tiene barreras
        float acc = 0.0f;
        for (int k = 0; k < n; ++k) acc += A[row * n + k] * B[k * n + col];
        C[row * n + col] = acc;
    }
}

// ---------- Kernel por mosaicos: 2·⌈n/TILE⌉ lecturas de memoria global por hilo ----------
__global__ void matmul_tiled(const float *A, const float *B, float *C, int n) {
    // Memoria compartida del bloque: dos mosaicos de TILE×TILE floats (2·TILE²·4 bytes; 2 KB con TILE = 16).
    // La ven todos los hilos del bloque y solo ellos; vive lo que vive el bloque.
    __shared__ float s_A[TILE][TILE];
    __shared__ float s_B[TILE][TILE];

    int tx = threadIdx.x, ty = threadIdx.y;
    int row = blockIdx.y * TILE + ty;  // fila de C que calcula este hilo
    int col = blockIdx.x * TILE + tx;  // columna de C que calcula este hilo
    float acc = 0.0f;                  // acumulador en un registro: privado de este hilo

    int n_tiles = (n + TILE - 1) / TILE;  // fases: una por mosaico a lo largo de k (redondeo hacia arriba)
    for (int t = 0; t < n_tiles; ++t) {
        // (1) Carga cooperativa: cada hilo trae UN elemento de A y UNO de B. Los hilos de un warp (tx
        //     consecutivos) leen direcciones consecutivas de A y de B: lecturas coalescidas.
        //     Fuera de la matriz (n no múltiplo de TILE) se carga 0.0f: ese producto parcial vale 0 y no hay
        //     que tratar los bordes aparte en el ciclo de productos.
        int a_col = t * TILE + tx;
        s_A[ty][tx] = (row < n && a_col < n) ? A[row * n + a_col] : 0.0f;
        int b_row = t * TILE + ty;
        s_B[ty][tx] = (b_row < n && col < n) ? B[b_row * n + col] : 0.0f;

        // (2) Primera barrera: nadie lee el mosaico hasta que TODOS los hilos del bloque terminaron de cargarlo
        //     (s_A[ty][k] lo trajo otro hilo). Sincroniza solo este bloque, no toda la GPU (idea errónea 25).
        __syncthreads();

        // (3) TILE productos con datos de memoria compartida: cada elemento cargado se usa TILE veces.
        for (int k = 0; k < TILE; ++k) acc += s_A[ty][k] * s_B[k][tx];

        // (4) Segunda barrera: nadie sobreescribe el mosaico (fase t+1) mientras otro hilo todavía lo lee.
        __syncthreads();
    }
    // (5) Guardia de escritura DESPUÉS del ciclo. No puede ser un "return" al inicio: los hilos fuera de rango
    //     también deben llegar a cada __syncthreads() y cargar sus ceros (ideas erróneas 32 y 34).
    if (row < n && col < n) C[row * n + col] = acc;
}

// ---------- Anfitrión ----------

// Inicialización determinista: la MISMA fórmula está en a_matmul_cpu.c, a_matmul_numba.py y en la Parte B.
static void init_matrices(float *A, float *B, int n) {
    for (int i = 0; i < n; ++i)
        for (int j = 0; j < n; ++j) {
            A[(size_t)i * n + j] = (float)((3 * i + 7 * j) % 11) / 11.0f;
            B[(size_t)i * n + j] = (float)((5 * i + 2 * j) % 13) / 13.0f;
        }
}

// Referencia secuencial, orden i-k-j (el ciclo interno recorre B y C por filas: amigable con la caché).
static void matmul_cpu(const float *A, const float *B, float *C, int n) {
    for (size_t idx = 0; idx < (size_t)n * n; ++idx) C[idx] = 0.0f;
    for (int i = 0; i < n; ++i)
        for (int k = 0; k < n; ++k) {
            float a = A[(size_t)i * n + k];
            for (int j = 0; j < n; ++j) C[(size_t)i * n + j] += a * B[(size_t)k * n + j];
        }
}

// Tolerancia: los sumandos están en [0,1) y cada C[i][j] vale ~0.2·n; el error de redondeo en float
// crece con n y queda muy por debajo de 1e-4·n. Un elemento sin calcular (error ~0.2·n) se detecta.
static float tolerance(int n) { return 1e-4f * (float)n; }

static double checksum_of(const float *x, size_t count) {
    double sum = 0.0;
    for (size_t i = 0; i < count; ++i) sum += x[i];
    return sum;
}

static float max_abs_diff(const float *x, const float *y, size_t count) {
    float m = 0.0f;
    for (size_t i = 0; i < count; ++i) {
        float e = fabsf(x[i] - y[i]);
        if (e > m) m = e;
    }
    return m;
}

// Mide H2D (A y B), kernel y D2H (C) con cuatro eventos; 1 calentamiento + REPS. Deja C en h_C.
static GpuTimes measure(int tiled, int n, dim3 grid, dim3 block, const float *h_A, const float *h_B,
                        float *h_C) {
    size_t bytes = (size_t)n * n * sizeof(float);
    float *d_A, *d_B, *d_C;
    CUDA_CHECK(cudaMalloc(&d_A, bytes));
    CUDA_CHECK(cudaMalloc(&d_B, bytes));
    CUDA_CHECK(cudaMalloc(&d_C, bytes));
    CUDA_CHECK(cudaMemset(d_C, 0, bytes));
    cudaEvent_t ev[4];
    for (int i = 0; i < 4; ++i) CUDA_CHECK(cudaEventCreate(&ev[i]));
    double h2d[REPS], ker[REPS], d2h[REPS];
    for (int r = -1; r < REPS; ++r) {  // r = -1 es el calentamiento (no se cuenta)
        CUDA_CHECK(cudaEventRecord(ev[0], 0));
        CUDA_CHECK(cudaMemcpy(d_A, h_A, bytes, cudaMemcpyHostToDevice));
        CUDA_CHECK(cudaMemcpy(d_B, h_B, bytes, cudaMemcpyHostToDevice));
        CUDA_CHECK(cudaEventRecord(ev[1], 0));
        if (tiled) matmul_tiled<<<grid, block>>>(d_A, d_B, d_C, n);
        else matmul_naive<<<grid, block>>>(d_A, d_B, d_C, n);
        CUDA_CHECK(cudaEventRecord(ev[2], 0));
        CUDA_CHECK(cudaMemcpy(h_C, d_C, bytes, cudaMemcpyDeviceToHost));
        CUDA_CHECK(cudaEventRecord(ev[3], 0));
        CUDA_CHECK(cudaEventSynchronize(ev[3]));  // sin esto se mide el encolado, no el trabajo (idea 33)
        // Con -DTILE=64 el bloque tendría 4096 hilos (> 1024): el lanzamiento falla y SOLO se nota aquí.
        CUDA_CHECK(cudaGetLastError());
        if (r >= 0) {
            float ms;
            CUDA_CHECK(cudaEventElapsedTime(&ms, ev[0], ev[1])); h2d[r] = ms;
            CUDA_CHECK(cudaEventElapsedTime(&ms, ev[1], ev[2])); ker[r] = ms;
            CUDA_CHECK(cudaEventElapsedTime(&ms, ev[2], ev[3])); d2h[r] = ms;
        }
    }
    GpuTimes t;
    t.h2d_ms = median_of(h2d, REPS);
    t.kernel_ms = median_of(ker, REPS);
    t.d2h_ms = median_of(d2h, REPS);
    for (int i = 0; i < 4; ++i) CUDA_CHECK(cudaEventDestroy(ev[i]));
    CUDA_CHECK(cudaFree(d_A));
    CUDA_CHECK(cudaFree(d_B));
    CUDA_CHECK(cudaFree(d_C));
    return t;
}

int main(int argc, char **argv) {
    int n = (argc > 1) ? atoi(argv[1]) : 512;
    const char *csv_path = (argc > 2) ? argv[2] : NULL;
    if (n <= 0) {
        fprintf(stderr, "Uso: %s [n>0] [salida.csv]   (TILE=%d; cambia con -DTILE=8|16|32)\n", argv[0], TILE);
        return 1;
    }
    size_t count = (size_t)n * n, bytes = count * sizeof(float);
    float *h_A = (float *)malloc(bytes), *h_B = (float *)malloc(bytes);
    float *h_C = (float *)malloc(bytes), *h_ref = (float *)malloc(bytes);
    if (!h_A || !h_B || !h_C || !h_ref) { fprintf(stderr, "Sin memoria en el anfitrión\n"); return 1; }
    init_matrices(h_A, h_B, n);

    cudaDeviceProp prop;
    char device[256];
    describe_device(&prop, device, sizeof device);

    // ---- T(1): referencia secuencial. 5 repeticiones hasta n = 1024; más allá tardaría minutos ----
    int cpu_reps = (n <= 1024) ? REPS : 1;
    double t_cpu[REPS];
    matmul_cpu(h_A, h_B, h_ref, n);  // calentamiento
    for (int r = 0; r < cpu_reps; ++r) {
        double t0 = cpu_seconds();
        matmul_cpu(h_A, h_B, h_ref, n);
        t_cpu[r] = (cpu_seconds() - t0) * 1e3;
    }
    double cpu_ms = median_of(t_cpu, cpu_reps);
    printf("matmul n=%d TILE=%d · CPU (i-k-j, -O2): mediana %.3f ms de %d repeticiones · checksum=%.6f\n",
           n, TILE, cpu_ms, cpu_reps, checksum_of(h_ref, count));

    FILE *csv = open_csv(csv_path);
    print_table_header();
    int failures = 0;
    double useful_bytes = 3.0 * (double)bytes;  // A, B y C una vez
    double flops = 2.0 * (double)n * n * n;

    // Malla 2D con redondeo hacia arriba en cada dimensión: x recorre columnas de C, y recorre filas.
    // Las dos versiones usan bloques de TILE×TILE hilos para que la comparación sea justa.
    int n_blocks = (n + TILE - 1) / TILE;
    dim3 block(TILE, TILE), grid(n_blocks, n_blocks);
    const char *names[2] = {"matmul_naive", "matmul_tiled"};
    for (int tiled = 0; tiled <= 1; ++tiled) {
        GpuTimes t = measure(tiled, n, grid, block, h_A, h_B, h_C);
        float err = max_abs_diff(h_C, h_ref, count);
        int ok = (err <= tolerance(n));
        failures += !ok;
        double occ = tiled ? occupancy_pct(matmul_tiled, TILE * TILE, 0, &prop)
                           : occupancy_pct(matmul_naive, TILE * TILE, 0, &prop);
        emit_row(csv, "bloque", names[tiled], n, TILE * TILE, (long)n_blocks * n_blocks, occ, "pageable", t,
                 cpu_ms, useful_bytes, ok, device);
        printf("    %s: %.1f GFLOP/s · error máximo %.2e (tolerancia %.2e) · checksum=%.6f", names[tiled],
               flops / (t.kernel_ms * 1e-3) / 1e9, err, tolerance(n), checksum_of(h_C, count));
        if (tiled) printf(" · fases=%d · compartida=%zu B/bloque", n_blocks, 2 * sizeof(float) * TILE * TILE);
        printf("\n");
    }

    if (csv) fclose(csv);
    printf("Verificación contra la CPU: %s\n", failures ? "FALLA" : "OK");
    free(h_A); free(h_B); free(h_C); free(h_ref);
    CUDA_CHECK(cudaDeviceReset());
    return failures ? 2 : 0;
}

In [ ]:
%%writefile a_matmul_cpu.c
/*
 * a_matmul_cpu.c — Referencia secuencial de la Parte A, multiplicación de matrices (C = A·B, n×n,
 * float), y calculadora del modelo de lecturas a memoria global de las dos versiones de GPU.
 *
 * Uso:      ./a_matmul_cpu [n=512] [tile=16]
 * Compilar: gcc -std=c11 -O2 -Wall -Wextra a_matmul_cpu.c -o a_matmul_cpu
 *
 * Misma inicialización que a_matmul_tiled.cu y a_matmul_numba.py: el checksum (suma de C en double)
 * sirve para comparar los tres programas. Protocolo: 1 calentamiento + 5 repeticiones (mediana) hasta
 * n = 1024; con n mayor una sola repetición (y se dice).
 */
#define _POSIX_C_SOURCE 200809L
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

#define REPS 5

static double now_s(void)
{
    struct timespec ts;
    clock_gettime(CLOCK_MONOTONIC, &ts);
    return (double)ts.tv_sec + (double)ts.tv_nsec * 1e-9;
}

static void init_matrices(float *A, float *B, int n)
{
    for (int i = 0; i < n; ++i)
        for (int j = 0; j < n; ++j) {
            A[(size_t)i * n + j] = (float)((3 * i + 7 * j) % 11) / 11.0f;
            B[(size_t)i * n + j] = (float)((5 * i + 2 * j) % 13) / 13.0f;
        }
}

/* Orden i-k-j: el ciclo interno recorre B y C por filas (amigable con la caché de la CPU). */
static void matmul_cpu(const float *A, const float *B, float *C, int n)
{
    for (size_t idx = 0; idx < (size_t)n * n; ++idx)
        C[idx] = 0.0f;
    for (int i = 0; i < n; ++i)
        for (int k = 0; k < n; ++k) {
            float a = A[(size_t)i * n + k];
            for (int j = 0; j < n; ++j)
                C[(size_t)i * n + j] += a * B[(size_t)k * n + j];
        }
}

static int cmp_double(const void *x, const void *y)
{
    double dx = *(const double *)x, dy = *(const double *)y;
    return (dx > dy) - (dx < dy);
}

int main(int argc, char **argv)
{
    int n = (argc > 1) ? atoi(argv[1]) : 512;
    int tile = (argc > 2) ? atoi(argv[2]) : 16;
    if (n <= 0 || tile <= 0) {
        fprintf(stderr, "uso: %s [n>0] [tile>0]\n", argv[0]);
        return 1;
    }
    size_t count = (size_t)n * n, bytes = count * sizeof(float);
    float *A = malloc(bytes), *B = malloc(bytes), *C = malloc(bytes);
    if (!A || !B || !C) {
        fprintf(stderr, "Sin memoria para n=%d\n", n);
        return 1;
    }
    init_matrices(A, B, n);

    int reps = (n <= 1024) ? REPS : 1;
    double t[REPS];
    matmul_cpu(A, B, C, n); /* calentamiento */
    for (int r = 0; r < reps; ++r) {
        double t0 = now_s();
        matmul_cpu(A, B, C, n);
        t[r] = now_s() - t0;
    }
    qsort(t, (size_t)reps, sizeof(double), cmp_double);
    double t_med = t[reps / 2];

    double checksum = 0.0;
    for (size_t i = 0; i < count; ++i)
        checksum += C[i];

    int phases = (n + tile - 1) / tile;
    unsigned long long reads_naive = 2ULL * (unsigned long long)n * n * n;
    unsigned long long reads_tiled = 2ULL * (unsigned long long)n * n * (unsigned long long)phases;
    double gflops = 2.0 * (double)n * n * n / t_med / 1e9;

    printf("a_matmul_cpu: n=%d tile=%d t_cpu_ms=%.3f (mediana de %d; min %.3f, max %.3f) %.2f GFLOP/s\n",
           n, tile, t_med * 1e3, reps, t[0] * 1e3, t[reps - 1] * 1e3, gflops);
    printf("  modelo de lecturas a memoria global: ingenua=%llu (cada elemento de A: %d veces) "
           "mosaicos=%llu (cada elemento de A: %d veces) cociente=%.2f fases=%d\n",
           reads_naive, n, reads_tiled, phases, (double)reads_naive / (double)reads_tiled, phases);
    printf("  C[0][0]=%.6f C[n-1][n-1]=%.6f checksum=%.6f\n", C[0], C[count - 1], checksum);

    free(A);
    free(B);
    free(C);
    return 0;
}

In [ ]:
%%writefile a_matmul_numba.py
#!/usr/bin/env python3
"""a_matmul_numba.py — Espejo en Numba CUDA de a_matmul_tiled.cu (Parte A, código completo): kernel ingenuo y
kernel por mosaicos con memoria compartida (cuda.shared.array) y dos barreras (cuda.syncthreads).
Sirve para ejecutar y rastrear la lógica SIN GPU:

    NUMBA_ENABLE_CUDASIM=1 python3 a_matmul_numba.py 32 16        # n=32, TILE=16 (≈ 5 s)
    NUMBA_ENABLE_CUDASIM=1 python3 a_matmul_numba.py 20 16        # n NO múltiplo de TILE

Uso: python3 a_matmul_numba.py [n=32 en CUDASIM / 512 en GPU] [tile=16] [naive|tiled|ambos]
Misma inicialización y mismo checksum que a_matmul_cpu.c. Verificación contra NumPy con la misma tolerancia
que el .cu (1e-4·n). Código de salida: 0 si todo PASS, 2 si algo falla. Bajo CUDASIM cada hilo se simula en
Python: usa n <= 64; los tiempos que imprime el simulador no significan nada.
"""
import os
import sys

import numpy as np
from numba import cuda, float32

CUDASIM = os.environ.get("NUMBA_ENABLE_CUDASIM", "0") == "1"
REPS = 5


@cuda.jit
def matmul_naive(A, B, C, n):
    row = cuda.blockIdx.y * cuda.blockDim.y + cuda.threadIdx.y
    col = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
    if row < n and col < n:  # este kernel no tiene barreras: la guardia puede envolver todo
        acc = float32(0.0)
        for k in range(n):
            acc += A[row, k] * B[k, col]
        C[row, col] = acc


def make_tiled_kernel(tile):
    """El tamaño de cuda.shared.array debe ser constante al compilar: por eso el kernel se construye dentro
    de una función que captura `tile` (en CUDA C es la macro TILE)."""

    @cuda.jit
    def matmul_tiled(A, B, C, n):
        s_A = cuda.shared.array(shape=(tile, tile), dtype=float32)
        s_B = cuda.shared.array(shape=(tile, tile), dtype=float32)
        tx = cuda.threadIdx.x
        ty = cuda.threadIdx.y
        row = cuda.blockIdx.y * tile + ty
        col = cuda.blockIdx.x * tile + tx
        acc = float32(0.0)
        n_tiles = (n + tile - 1) // tile
        for t in range(n_tiles):
            # (1) carga cooperativa con cero fuera de la matriz
            a_col = t * tile + tx
            if row < n and a_col < n:
                s_A[ty, tx] = A[row, a_col]
            else:
                s_A[ty, tx] = float32(0.0)
            b_row = t * tile + ty
            if b_row < n and col < n:
                s_B[ty, tx] = B[b_row, col]
            else:
                s_B[ty, tx] = float32(0.0)
            cuda.syncthreads()  # (2) el mosaico completo antes de leerlo
            for k in range(tile):  # (3) tile productos con datos de memoria compartida
                acc += s_A[ty, k] * s_B[k, tx]
            cuda.syncthreads()  # (4) nadie sobreescribe el mosaico mientras otro lo lee
        if row < n and col < n:  # (5) guardia de escritura después del ciclo, no "return" al inicio
            C[row, col] = acc

    return matmul_tiled


def init_matrices(n):
    # Misma fórmula que a_matmul_tiled.cu y a_matmul_cpu.c.
    i = np.arange(n).reshape(n, 1)
    j = np.arange(n).reshape(1, n)
    A = (((3 * i + 7 * j) % 11) / 11.0).astype(np.float32)
    B = (((5 * i + 2 * j) % 13) / 13.0).astype(np.float32)
    return A, B


def tolerance(n):
    return 1e-4 * n


def median(v):
    return sorted(v)[len(v) // 2]


def run(name, kernel, A, B, n, tile):
    grid_dim = (n + tile - 1) // tile  # redondeo hacia arriba en cada dimensión
    C = np.zeros((n, n), dtype=np.float32)
    d_A, d_B, d_C = cuda.to_device(A), cuda.to_device(B), cuda.to_device(C)
    kernel[(grid_dim, grid_dim), (tile, tile)](d_A, d_B, d_C, n)  # calentamiento (JIT en GPU real)
    cuda.synchronize()
    times = []
    for _ in range(1 if CUDASIM else REPS):  # en el simulador el tiempo no es una medición
        e0, e1 = cuda.event(), cuda.event()
        e0.record()
        kernel[(grid_dim, grid_dim), (tile, tile)](d_A, d_B, d_C, n)
        e1.record()
        e1.synchronize()  # el lanzamiento es asíncrono: sin esto se mide casi nada
        times.append(e0.elapsed_time(e1))
    d_C.copy_to_host(C)
    ref = A.astype(np.float64) @ B.astype(np.float64)
    err = float(np.max(np.abs(C - ref)))
    checksum = float(np.sum(C, dtype=np.float64))
    ok = err <= tolerance(n)
    print(f"{name:<13} n={n} tile={tile} malla={grid_dim}x{grid_dim} bloque={tile}x{tile} fases={grid_dim} "
          f"t_kernel_ms={median(times):.3f}{' (CUDASIM: no representativo)' if CUDASIM else ''}")
    print(f"    error maximo vs NumPy {err:.2e} (tolerancia {tolerance(n):.1e}) checksum={checksum:.6f} "
          f"verificacion={'PASS' if ok else 'FAIL'}")
    return ok


def main():
    n = int(sys.argv[1]) if len(sys.argv) > 1 else (32 if CUDASIM else 512)
    tile = int(sys.argv[2]) if len(sys.argv) > 2 else 16
    which = sys.argv[3] if len(sys.argv) > 3 else "ambos"
    if n <= 0 or tile <= 0 or tile * tile > 1024 or which not in ("naive", "tiled", "ambos"):
        print("uso: a_matmul_numba.py [n>0] [tile; tile*tile <= 1024] [naive|tiled|ambos]")
        sys.exit(1)
    if CUDASIM and n > 64:
        print("aviso: CUDASIM con n > 64 tarda minutos", file=sys.stderr)
    A, B = init_matrices(n)
    ok = True
    if which != "tiled":
        ok &= run("matmul_naive", matmul_naive, A, B, n, tile)
    if which != "naive":
        ok &= run("matmul_tiled", make_tiled_kernel(tile), A, B, n, tile)
    sys.exit(0 if ok else 2)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile b_matmul_rect.cu
// b_matmul_rect.cu — Práctica 6 · Parte B (semi-guiada): multiplicación por mosaicos con matrices
// RECTANGULARES. C = A·B con A de M×K, B de K×N y C de M×N (float, guardadas por filas).
//   matmul_rect_naive  COMPLETA: un hilo por elemento de C, lee de memoria global. Úsala como referencia
//                      de índices: ahí está cómo se recorre cada matriz con su número de columnas.
//   matmul_rect_tiled  la versión por mosaicos de la Parte A (a_matmul_tiled.cu) con n cambiado por M en
//                      todos lados: está "generalizada a medias". Las cuatro líneas marcadas TODO(1)–TODO(4)
//                      son correctas cuando M = K = N y están mal cuando las matrices son rectangulares.
//                      La memoria compartida, las dos __syncthreads() y el ciclo de productos YA están
//                      escritos y no cambian: son los que leíste y rastreaste en la Parte A.
//
// Compilar (GPU):     nvcc -O2 -arch=sm_75 b_matmul_rect.cu -o b_matmul_rect        (-DTILE=8|16|32)
// Verificar sin GPU:  nvcc -c -arch=sm_75 b_matmul_rect.cu -o b_matmul_rect.o
//                     y el espejo: NUMBA_ENABLE_CUDASIM=1 python3 b_matmul_rect_numba.py 20 40 12 16
// Uso:                ./b_matmul_rect [M=1000] [K=600] [N=700] [salida.csv]
//
// Tu trabajo: corrige TODO(1)–TODO(4) (aquí y en b_matmul_rect_numba.py) y pon TODO_LISTO en 1. Mientras
// TODO_LISTO valga 0 el programa imprime "TODO pendiente" y termina con código 3 sin lanzar el kernel por
// mosaicos (la ingenua y la CPU sí corren). Si pones TODO_LISTO en 1 sin corregir nada, la prueba con
// M = K = N = 32 PASA y la de 20×40×12 FALLA: una prueba que pasa no demuestra que el código sea correcto
// (ideas erróneas 27 y 34). Verificación: con tolerancia 1e-4·K por elemento. Medición: 1 calentamiento +
// 5 repeticiones de H2D + kernel + D2H, mediana de cada tramo. CSV: experimento "uno", n = M·N (elementos de C).
#include "common.cuh"

#ifndef TILE
#define TILE 16
#endif

// Pon 1 cuando hayas corregido TODO(1)–TODO(4). Mientras valga 0, el programa imprime "TODO pendiente"
// y no lanza el kernel por mosaicos (la ingenua y la CPU sí corren).
#define TODO_LISTO 0

// ---------- Kernel ingenuo rectangular (completo) ----------
__global__ void matmul_rect_naive(const float *A, const float *B, float *C, int M, int K, int N) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;  // fila de C: 0 .. M-1
    int col = blockIdx.x * blockDim.x + threadIdx.x;  // columna de C: 0 .. N-1
    if (row < M && col < N) {  // sin barreras en este kernel: la guardia puede envolver todo
        float acc = 0.0f;
        for (int k = 0; k < K; ++k) acc += A[row * K + k] * B[k * N + col];  // A tiene K columnas; B, N
        C[row * N + col] = acc;
    }
}

// TODO(1): regresa mat[r][c] de una matriz de rows×cols guardada por filas, o 0.0f si (r, c) cae fuera
//          (r >= rows o c >= cols). Dos cosas de esta línea solo valen para matrices cuadradas: ¿cuántos
//          elementos tiene una FILA de la matriz? ¿contra qué se compara la columna c?
__device__ float load_or_zero(const float *mat, int r, int c, int rows, int cols) {
    return (r < rows && c < rows) ? mat[r * rows + c] : 0.0f; /* <- TODO(1) */
}

// ---------- Kernel por mosaicos rectangular: corrige TODO(2)–TODO(4) ----------
__global__ void matmul_rect_tiled(const float *A, const float *B, float *C, int M, int K, int N) {
    __shared__ float s_A[TILE][TILE];  // mosaico de A: TILE filas de C × TILE posiciones de k
    __shared__ float s_B[TILE][TILE];  // mosaico de B: TILE posiciones de k × TILE columnas de C

    int tx = threadIdx.x, ty = threadIdx.y;
    int row = blockIdx.y * TILE + ty;  // fila de C (y de A) que calcula este hilo
    int col = blockIdx.x * TILE + tx;  // columna de C (y de B) que calcula este hilo
    float acc = 0.0f;

    // TODO(2): ¿a lo largo de qué dimensión avanzan las fases? En C[row][col] = Σ_k A[row][k]·B[k][col],
    //          ¿hasta dónde llega k?
    int n_tiles = (M + TILE - 1) / TILE; /* <- TODO(2) */
    for (int t = 0; t < n_tiles; ++t) {
        // TODO(3): los índices (fila, columna) que trae cada hilo son los de la Parte A; lo que falta es el
        //          tamaño (filas, columnas) de cada matriz en los dos últimos argumentos.
        s_A[ty][tx] = load_or_zero(A, row, t * TILE + tx, M, M); /* <- TODO(3) */
        s_B[ty][tx] = load_or_zero(B, t * TILE + ty, col, M, M); /* <- TODO(3) */
        __syncthreads();  // (dado) el mosaico completo está en memoria compartida antes de leerlo
        for (int k = 0; k < TILE; ++k) acc += s_A[ty][k] * s_B[k][tx];
        __syncthreads();  // (dado) nadie sobreescribe el mosaico mientras otro hilo todavía lo lee
    }
    // TODO(4): esta línea tiene dos defectos que con matrices cuadradas no se notan.
    if (row < M && col < M) C[row * M + col] = acc; /* <- TODO(4) */
}

// ---------- Anfitrión ----------

// Inicialización determinista: la de la Parte A con otras formas (igual en b_matmul_rect_cpu.c y el espejo).
static void init_matrices(float *A, float *B, int M, int K, int N) {
    for (int i = 0; i < M; ++i)
        for (int j = 0; j < K; ++j) A[(size_t)i * K + j] = (float)((3 * i + 7 * j) % 11) / 11.0f;
    for (int i = 0; i < K; ++i)
        for (int j = 0; j < N; ++j) B[(size_t)i * N + j] = (float)((5 * i + 2 * j) % 13) / 13.0f;
}

static void matmul_rect_cpu(const float *A, const float *B, float *C, int M, int K, int N) {
    for (size_t idx = 0; idx < (size_t)M * N; ++idx) C[idx] = 0.0f;
    for (int i = 0; i < M; ++i)
        for (int k = 0; k < K; ++k) {
            float a = A[(size_t)i * K + k];
            for (int j = 0; j < N; ++j) C[(size_t)i * N + j] += a * B[(size_t)k * N + j];
        }
}

// Cada C[i][j] suma K productos en [0,1): vale ~0.2·K. Un elemento sin calcular (error ~0.2·K) se detecta.
static float tolerance(int K) { return 1e-4f * (float)K; }

static double checksum_of(const float *x, size_t count) {
    double sum = 0.0;
    for (size_t i = 0; i < count; ++i) sum += x[i];
    return sum;
}

static float max_abs_diff(const float *x, const float *y, size_t count) {
    float m = 0.0f;
    for (size_t i = 0; i < count; ++i) {
        float e = fabsf(x[i] - y[i]);
        if (e > m) m = e;
    }
    return m;
}

// Mide H2D (A y B), kernel y D2H (C) con cuatro eventos; 1 calentamiento + REPS. Deja C en h_C.
static GpuTimes measure(int tiled, int M, int K, int N, dim3 grid, dim3 block, const float *h_A,
                        const float *h_B, float *h_C) {
    size_t bytes_A = (size_t)M * K * sizeof(float), bytes_B = (size_t)K * N * sizeof(float);
    size_t bytes_C = (size_t)M * N * sizeof(float);
    float *d_A, *d_B, *d_C;
    CUDA_CHECK(cudaMalloc(&d_A, bytes_A));
    CUDA_CHECK(cudaMalloc(&d_B, bytes_B));
    CUDA_CHECK(cudaMalloc(&d_C, bytes_C));
    CUDA_CHECK(cudaMemset(d_C, 0, bytes_C));  // un elemento que nadie escribe queda en 0 y la verificación lo ve
    cudaEvent_t ev[4];
    for (int i = 0; i < 4; ++i) CUDA_CHECK(cudaEventCreate(&ev[i]));
    double h2d[REPS], ker[REPS], d2h[REPS];
    for (int r = -1; r < REPS; ++r) {  // r = -1 es el calentamiento (no se cuenta)
        CUDA_CHECK(cudaEventRecord(ev[0], 0));
        CUDA_CHECK(cudaMemcpy(d_A, h_A, bytes_A, cudaMemcpyHostToDevice));
        CUDA_CHECK(cudaMemcpy(d_B, h_B, bytes_B, cudaMemcpyHostToDevice));
        CUDA_CHECK(cudaEventRecord(ev[1], 0));
        if (tiled) matmul_rect_tiled<<<grid, block>>>(d_A, d_B, d_C, M, K, N);
        else matmul_rect_naive<<<grid, block>>>(d_A, d_B, d_C, M, K, N);
        CUDA_CHECK(cudaEventRecord(ev[2], 0));
        CUDA_CHECK(cudaMemcpy(h_C, d_C, bytes_C, cudaMemcpyDeviceToHost));
        CUDA_CHECK(cudaEventRecord(ev[3], 0));
        CUDA_CHECK(cudaEventSynchronize(ev[3]));
        CUDA_CHECK(cudaGetLastError());  // errores de lanzamiento (p. ej. -DTILE=64: 4096 hilos por bloque)
        if (r >= 0) {
            float ms;
            CUDA_CHECK(cudaEventElapsedTime(&ms, ev[0], ev[1])); h2d[r] = ms;
            CUDA_CHECK(cudaEventElapsedTime(&ms, ev[1], ev[2])); ker[r] = ms;
            CUDA_CHECK(cudaEventElapsedTime(&ms, ev[2], ev[3])); d2h[r] = ms;
        }
    }
    GpuTimes t;
    t.h2d_ms = median_of(h2d, REPS);
    t.kernel_ms = median_of(ker, REPS);
    t.d2h_ms = median_of(d2h, REPS);
    for (int i = 0; i < 4; ++i) CUDA_CHECK(cudaEventDestroy(ev[i]));
    CUDA_CHECK(cudaFree(d_A));
    CUDA_CHECK(cudaFree(d_B));
    CUDA_CHECK(cudaFree(d_C));
    return t;
}

int main(int argc, char **argv) {
    int M = (argc > 1) ? atoi(argv[1]) : 1000;
    int K = (argc > 2) ? atoi(argv[2]) : 600;
    int N = (argc > 3) ? atoi(argv[3]) : 700;
    const char *csv_path = (argc > 4) ? argv[4] : NULL;
    if (M <= 0 || K <= 0 || N <= 0) {
        fprintf(stderr, "Uso: %s [M>0] [K>0] [N>0] [salida.csv]   (TILE=%d; cambia con -DTILE=8|16|32)\n",
                argv[0], TILE);
        return 1;
    }
    size_t count_C = (size_t)M * N;
    float *h_A = (float *)malloc((size_t)M * K * sizeof(float));
    float *h_B = (float *)malloc((size_t)K * N * sizeof(float));
    float *h_C = (float *)malloc(count_C * sizeof(float)), *h_ref = (float *)malloc(count_C * sizeof(float));
    if (!h_A || !h_B || !h_C || !h_ref) { fprintf(stderr, "Sin memoria en el anfitrión\n"); return 1; }
    init_matrices(h_A, h_B, M, K, N);

    cudaDeviceProp prop;
    char device[256];
    describe_device(&prop, device, sizeof device);

    // ---- T(1): referencia secuencial (5 repeticiones hasta M·K·N = 2^30; más allá, una) ----
    int cpu_reps = ((double)M * K * N <= 1073741824.0) ? REPS : 1;
    double t_cpu[REPS];
    matmul_rect_cpu(h_A, h_B, h_ref, M, K, N);  // calentamiento
    for (int r = 0; r < cpu_reps; ++r) {
        double t0 = cpu_seconds();
        matmul_rect_cpu(h_A, h_B, h_ref, M, K, N);
        t_cpu[r] = (cpu_seconds() - t0) * 1e3;
    }
    double cpu_ms = median_of(t_cpu, cpu_reps);
    printf("matmul_rect M=%d K=%d N=%d TILE=%d · CPU (i-k-j, -O2): mediana %.3f ms de %d · checksum=%.6f\n", M,
           K, N, TILE, cpu_ms, cpu_reps, checksum_of(h_ref, count_C));

    // (Dado) Malla 2D para C (M×N): x recorre las N columnas, y recorre las M filas; redondeo hacia arriba.
    dim3 block(TILE, TILE), grid((N + TILE - 1) / TILE, (M + TILE - 1) / TILE);
    long n_blocks = (long)grid.x * grid.y;
    double useful_bytes = 4.0 * ((double)M * K + (double)K * N + (double)M * N);  // A, B y C una vez
    double flops = 2.0 * (double)M * K * N;

    FILE *csv = open_csv(csv_path);
    print_table_header();
    int failures = 0;
    const char *names[2] = {"matmul_rect_naive", "matmul_rect_tiled"};
    for (int tiled = 0; tiled <= 1; ++tiled) {
#if !TODO_LISTO
        if (tiled) {
            printf("TODO pendiente: corrige TODO(1)-TODO(4) en b_matmul_rect.cu y pon TODO_LISTO en 1\n");
            if (csv) fclose(csv);
            free(h_A); free(h_B); free(h_C); free(h_ref);
            return 3;
        }
#endif
        GpuTimes t = measure(tiled, M, K, N, grid, block, h_A, h_B, h_C);
        float err = max_abs_diff(h_C, h_ref, count_C);
        int ok = (err <= tolerance(K));
        failures += !ok;
        double occ = tiled ? occupancy_pct(matmul_rect_tiled, TILE * TILE, 0, &prop)
                           : occupancy_pct(matmul_rect_naive, TILE * TILE, 0, &prop);
        emit_row(csv, "uno", names[tiled], (long)count_C, TILE * TILE, n_blocks, occ, "pageable", t, cpu_ms,
                 useful_bytes, ok, device);
        printf("    %s: %.1f GFLOP/s · error máximo %.2e (tolerancia %.2e) · checksum=%.6f\n", names[tiled],
               flops / (t.kernel_ms * 1e-3) / 1e9, err, tolerance(K), checksum_of(h_C, count_C));
    }

    if (csv) fclose(csv);
    printf("Verificación contra la CPU: %s\n", failures ? "FALLA" : "OK");
    free(h_A); free(h_B); free(h_C); free(h_ref);
    CUDA_CHECK(cudaDeviceReset());
    return failures ? 2 : 0;
}

In [ ]:
%%writefile b_matmul_rect_cpu.c
/*
 * b_matmul_rect_cpu.c — Referencia secuencial de la Parte B: C = A·B con matrices RECTANGULARES
 * (A es M×K, B es K×N, C es M×N; float; todas guardadas por filas).
 *
 * Uso:      ./b_matmul_rect_cpu [M=1000] [K=600] [N=700] [tile=16]
 * Compilar: gcc -std=c11 -O2 -Wall -Wextra b_matmul_rect_cpu.c -o b_matmul_rect_cpu
 *
 * Misma inicialización que b_matmul_rect.cu y b_matmul_rect_numba.py (y que la Parte A cuando
 * M = K = N): el checksum (suma de C en double) sirve para comparar los tres programas.
 * Además imprime la geometría que debe tener la versión por mosaicos para ese tamaño: malla (x recorre
 * las N columnas de C, y recorre las M filas), fases a lo largo de K e hilos que no escriben en C.
 * Protocolo: 1 calentamiento + 5 repeticiones (mediana) hasta M·K·N = 2^30; más allá una sola (y se dice).
 */
#define _POSIX_C_SOURCE 200809L
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

#define REPS 5

static double now_s(void)
{
    struct timespec ts;
    clock_gettime(CLOCK_MONOTONIC, &ts);
    return (double)ts.tv_sec + (double)ts.tv_nsec * 1e-9;
}

/* A[i][j] con i < M, j < K y B[i][j] con i < K, j < N: las fórmulas de la Parte A con otras formas. */
static void init_matrices(float *A, float *B, int M, int K, int N)
{
    for (int i = 0; i < M; ++i)
        for (int j = 0; j < K; ++j)
            A[(size_t)i * K + j] = (float)((3 * i + 7 * j) % 11) / 11.0f;
    for (int i = 0; i < K; ++i)
        for (int j = 0; j < N; ++j)
            B[(size_t)i * N + j] = (float)((5 * i + 2 * j) % 13) / 13.0f;
}

/* Orden i-k-j: el ciclo interno recorre una fila de B y una de C (contiguas en memoria). */
static void matmul_rect_cpu(const float *A, const float *B, float *C, int M, int K, int N)
{
    for (size_t idx = 0; idx < (size_t)M * N; ++idx)
        C[idx] = 0.0f;
    for (int i = 0; i < M; ++i)
        for (int k = 0; k < K; ++k) {
            float a = A[(size_t)i * K + k];
            for (int j = 0; j < N; ++j)
                C[(size_t)i * N + j] += a * B[(size_t)k * N + j];
        }
}

static int cmp_double(const void *x, const void *y)
{
    double dx = *(const double *)x, dy = *(const double *)y;
    return (dx > dy) - (dx < dy);
}

int main(int argc, char **argv)
{
    int M = (argc > 1) ? atoi(argv[1]) : 1000;
    int K = (argc > 2) ? atoi(argv[2]) : 600;
    int N = (argc > 3) ? atoi(argv[3]) : 700;
    int tile = (argc > 4) ? atoi(argv[4]) : 16;
    if (M <= 0 || K <= 0 || N <= 0 || tile <= 0) {
        fprintf(stderr, "uso: %s [M>0] [K>0] [N>0] [tile>0]\n", argv[0]);
        return 1;
    }
    float *A = malloc((size_t)M * K * sizeof(float));
    float *B = malloc((size_t)K * N * sizeof(float));
    float *C = malloc((size_t)M * N * sizeof(float));
    if (!A || !B || !C) {
        fprintf(stderr, "Sin memoria para M=%d K=%d N=%d\n", M, K, N);
        return 1;
    }
    init_matrices(A, B, M, K, N);

    double work = (double)M * K * N;
    int reps = (work <= 1073741824.0) ? REPS : 1;
    double t[REPS];
    matmul_rect_cpu(A, B, C, M, K, N); /* calentamiento */
    for (int r = 0; r < reps; ++r) {
        double t0 = now_s();
        matmul_rect_cpu(A, B, C, M, K, N);
        t[r] = now_s() - t0;
    }
    qsort(t, (size_t)reps, sizeof(double), cmp_double);
    double t_med = t[reps / 2];

    double checksum = 0.0;
    for (size_t i = 0; i < (size_t)M * N; ++i)
        checksum += C[i];

    int grid_x = (N + tile - 1) / tile, grid_y = (M + tile - 1) / tile, phases = (K + tile - 1) / tile;
    long long launched = (long long)grid_x * grid_y * tile * tile;
    printf("b_matmul_rect_cpu: M=%d K=%d N=%d tile=%d t_cpu_ms=%.3f (mediana de %d; min %.3f, max %.3f) "
           "%.2f GFLOP/s\n",
           M, K, N, tile, t_med * 1e3, reps, t[0] * 1e3, t[reps - 1] * 1e3, 2.0 * work / t_med / 1e9);
    printf("  geometria por mosaicos: malla=%dx%d (x: columnas N, y: filas M) fases=%d (a lo largo de K) "
           "hilos=%lld sin_escritura=%lld\n",
           grid_x, grid_y, phases, launched, launched - (long long)M * N);
    printf("  C[0][0]=%.6f C[M-1][N-1]=%.6f checksum=%.6f\n", C[0], C[(size_t)M * N - 1], checksum);

    free(A);
    free(B);
    free(C);
    return 0;
}

In [ ]:
%%writefile b_matmul_rect_numba.py
#!/usr/bin/env python3
"""b_matmul_rect_numba.py — Espejo en Numba CUDA de b_matmul_rect.cu (Parte B): C = A·B con A de M×K, B de K×N
y C de M×N. Kernel ingenuo rectangular completo y kernel por mosaicos con los MISMOS TODO(1)–TODO(4) que el .cu.
Las matrices viajan como arreglos 1D guardados por filas (igual que en CUDA C), para que los índices lineales
signifiquen lo mismo en los dos programas. Sirve para verificar tu lógica SIN GPU:

    NUMBA_ENABLE_CUDASIM=1 python3 b_matmul_rect_numba.py 32 32 32 16     # cuadrada: pasa aun sin corregir
    NUMBA_ENABLE_CUDASIM=1 python3 b_matmul_rect_numba.py 20 40 12 16     # rectangular, no múltiplo de TILE

Uso: python3 b_matmul_rect_numba.py [M K N] [tile=16] [naive|tiled|ambos]   (por defecto 20 40 12 en CUDASIM)
Misma inicialización y mismo checksum que b_matmul_rect_cpu.c. Verificación contra NumPy con tolerancia 1e-4·K.
Código de salida: 0 si todo PASS, 2 si algo falla (o hay un acceso fuera de rango), 3 si hay TODO pendiente.
Bajo CUDASIM cada hilo se simula en Python: usa M, K, N <= 64.

Tu trabajo: corrige TODO(1)–TODO(4) (los mismos que en el .cu) y pon TODO_LISTO = True. Mientras sea False el
programa imprime "TODO pendiente" y termina con código 3 sin lanzar el kernel por mosaicos.
"""
import os
import sys

import numpy as np
from numba import cuda, float32

CUDASIM = os.environ.get("NUMBA_ENABLE_CUDASIM", "0") == "1"
REPS = 5

# Pon True cuando hayas corregido TODO(1)–TODO(4) (aquí y en el .cu).
TODO_LISTO = False


@cuda.jit
def matmul_rect_naive(A, B, C, M, K, N):
    row = cuda.blockIdx.y * cuda.blockDim.y + cuda.threadIdx.y
    col = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
    if row < M and col < N:  # sin barreras: la guardia puede envolver todo
        acc = float32(0.0)
        for k in range(K):
            acc += A[row * K + k] * B[k * N + col]  # A tiene K columnas; B, N
        C[row * N + col] = acc


@cuda.jit(device=True)
def load_or_zero(mat, r, c, rows, cols):
    # TODO(1): mat[r][c] de una matriz rows×cols guardada por filas, o 0.0 si (r, c) cae fuera.
    #          Dos cosas de estas líneas solo valen para matrices cuadradas.
    if r < rows and c < rows:  # <- TODO(1)
        return mat[r * rows + c]  # <- TODO(1)
    return float32(0.0)


def make_tiled_kernel(tile):
    """cuda.shared.array necesita un tamaño constante al compilar: el kernel se construye para cada `tile`."""

    @cuda.jit
    def matmul_rect_tiled(A, B, C, M, K, N):
        s_A = cuda.shared.array(shape=(tile, tile), dtype=float32)
        s_B = cuda.shared.array(shape=(tile, tile), dtype=float32)
        tx = cuda.threadIdx.x
        ty = cuda.threadIdx.y
        row = cuda.blockIdx.y * tile + ty  # fila de C (y de A)
        col = cuda.blockIdx.x * tile + tx  # columna de C (y de B)
        acc = float32(0.0)
        # TODO(2): ¿a lo largo de qué dimensión avanzan las fases?
        n_tiles = (M + tile - 1) // tile  # <- TODO(2)
        for t in range(n_tiles):
            # TODO(3): los índices son los de la Parte A; faltan las dimensiones (filas, columnas) de cada matriz.
            s_A[ty, tx] = load_or_zero(A, row, t * tile + tx, M, M)  # <- TODO(3)
            s_B[ty, tx] = load_or_zero(B, t * tile + ty, col, M, M)  # <- TODO(3)
            cuda.syncthreads()  # (dado) el mosaico completo antes de leerlo
            for k in range(tile):
                acc += s_A[ty, k] * s_B[k, tx]
            cuda.syncthreads()  # (dado) nadie sobreescribe el mosaico mientras otro lo lee
        # TODO(4): dos defectos que con matrices cuadradas no se notan.
        if row < M and col < M:  # <- TODO(4)
            C[row * M + col] = acc  # <- TODO(4)

    return matmul_rect_tiled


def init_matrices(M, K, N):
    # Las fórmulas de la Parte A con otras formas (igual que b_matmul_rect.cu y b_matmul_rect_cpu.c).
    i = np.arange(M).reshape(M, 1)
    j = np.arange(K).reshape(1, K)
    A = (((3 * i + 7 * j) % 11) / 11.0).astype(np.float32)
    i = np.arange(K).reshape(K, 1)
    j = np.arange(N).reshape(1, N)
    B = (((5 * i + 2 * j) % 13) / 13.0).astype(np.float32)
    return A, B


def median(v):
    return sorted(v)[len(v) // 2]


def run(name, kernel, A, B, M, K, N, tile):
    # (Dado) malla 2D para C (M×N): x recorre las N columnas, y recorre las M filas; redondeo hacia arriba.
    grid = ((N + tile - 1) // tile, (M + tile - 1) // tile)
    C = np.zeros(M * N, dtype=np.float32)
    d_A, d_B, d_C = cuda.to_device(A.ravel()), cuda.to_device(B.ravel()), cuda.to_device(C)
    times = []
    try:
        kernel[grid, (tile, tile)](d_A, d_B, d_C, M, K, N)  # calentamiento (JIT en GPU real)
        cuda.synchronize()
        for _ in range(1 if CUDASIM else REPS):  # en el simulador el tiempo no es una medición
            e0, e1 = cuda.event(), cuda.event()
            e0.record()
            kernel[grid, (tile, tile)](d_A, d_B, d_C, M, K, N)
            e1.record()
            e1.synchronize()
            times.append(e0.elapsed_time(e1))
    except IndexError as exc:  # CUDASIM revisa los límites de cada acceso; la GPU real no (idea errónea 34)
        print(f"{name:<17} M={M} K={K} N={N} tile={tile}: acceso fuera de rango en el kernel -> {exc}")
        print("    verificacion=FAIL")
        return False
    d_C.copy_to_host(C)
    ref = (A.astype(np.float64) @ B.astype(np.float64)).ravel()
    err = float(np.max(np.abs(C - ref)))
    checksum = float(np.sum(C, dtype=np.float64))
    ok = err <= 1e-4 * K
    print(f"{name:<17} M={M} K={K} N={N} tile={tile} malla={grid[0]}x{grid[1]} bloque={tile}x{tile} "
          f"t_kernel_ms={median(times):.3f}{' (CUDASIM: no representativo)' if CUDASIM else ''}")
    print(f"    error maximo vs NumPy {err:.2e} (tolerancia {1e-4 * K:.1e}) checksum={checksum:.6f} "
          f"verificacion={'PASS' if ok else 'FAIL'}")
    return ok


def main():
    args = sys.argv[1:]
    which = "ambos"
    if args and args[-1] in ("naive", "tiled", "ambos"):
        which = args.pop()
    dims = [int(a) for a in args[:3]] if len(args) >= 3 else ([20, 40, 12] if CUDASIM else [1000, 600, 700])
    tile = int(args[3]) if len(args) > 3 else 16
    M, K, N = dims
    if min(M, K, N) <= 0 or tile <= 0 or tile * tile > 1024:
        print("uso: b_matmul_rect_numba.py [M K N] [tile; tile*tile <= 1024] [naive|tiled|ambos]")
        sys.exit(1)
    if CUDASIM and max(M, K, N) > 64:
        print("aviso: CUDASIM con dimensiones > 64 tarda minutos", file=sys.stderr)
    A, B = init_matrices(M, K, N)
    ok = True
    if which != "tiled":
        ok &= run("matmul_rect_naive", matmul_rect_naive, A, B, M, K, N, tile)
    if which != "naive":
        if not TODO_LISTO:
            print("matmul_rect_tiled TODO pendiente: corrige TODO(1)-TODO(4) y pon TODO_LISTO = True")
            sys.exit(3)
        ok &= run("matmul_rect_tiled", make_tiled_kernel(tile), A, B, M, K, N, tile)
    sys.exit(0 if ok else 2)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile c_histograma_cpu.c
/*
 * c_histograma_cpu.c — Referencia secuencial de la Parte C, opción 1: histograma de 256 clases y
 * reducción (suma) sobre n bytes pseudoaleatorios.
 *
 * Uso:      ./c_histograma_cpu [n=16777216] [--volcar hist.txt]
 * Compilar: gcc -std=c11 -O2 -Wall -Wextra c_histograma_cpu.c -o c_histograma_cpu
 *
 * ENTRADA (la misma que debe usar tu kernel; el generador está también en common.cuh):
 *   data[i] = ((i * 2654435761) mod 2^32) >> 24          (byte en 0..255, hash multiplicativo de Knuth)
 * SALIDA que tu programa de GPU debe reproducir EXACTAMENTE (son enteros: aquí no hay tolerancia):
 *   suma            = Σ data[i]                            (la reducción)
 *   hist_checksum   = Σ_b (b + 1) · hist[b]                 (resume las 256 cuentas en un número)
 *   bin_max, cuenta_max                                     (clase más poblada y su cuenta)
 *   con --volcar se escriben las 256 cuentas (una por línea: "b,cuenta") para comparar completo.
 * T(1): 1 calentamiento + 5 repeticiones de (histograma + suma), mediana.
 */
#define _POSIX_C_SOURCE 200809L
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>

#define REPS 5
#define BINS 256

static double now_s(void)
{
    struct timespec ts;
    clock_gettime(CLOCK_MONOTONIC, &ts);
    return (double)ts.tv_sec + (double)ts.tv_nsec * 1e-9;
}

/* Idéntico a init_histogram_data() de common.cuh. */
static void init_data(unsigned char *data, long n)
{
    for (long i = 0; i < n; ++i) {
        unsigned int x = (unsigned int)i * 2654435761u; /* desborda a propósito (mod 2^32) */
        data[i] = (unsigned char)((x >> 24) & 0xFFu);
    }
}

static void histogram_cpu(const unsigned char *data, long n, unsigned int *hist)
{
    memset(hist, 0, BINS * sizeof(unsigned int));
    for (long i = 0; i < n; ++i)
        hist[data[i]]++;
}

static unsigned long long sum_cpu(const unsigned char *data, long n)
{
    unsigned long long s = 0;
    for (long i = 0; i < n; ++i)
        s += data[i];
    return s;
}

static int cmp_double(const void *x, const void *y)
{
    double dx = *(const double *)x, dy = *(const double *)y;
    return (dx > dy) - (dx < dy);
}

int main(int argc, char **argv)
{
    long n = (argc > 1 && argv[1][0] != '-') ? atol(argv[1]) : (1L << 24);
    const char *dump = NULL;
    for (int i = 1; i < argc - 1; ++i)
        if (strcmp(argv[i], "--volcar") == 0)
            dump = argv[i + 1];
    if (n <= 0) {
        fprintf(stderr, "uso: %s [n>0] [--volcar hist.txt]\n", argv[0]);
        return 1;
    }
    unsigned char *data = malloc((size_t)n);
    if (!data) {
        fprintf(stderr, "Sin memoria para n=%ld\n", n);
        return 1;
    }
    init_data(data, n);

    unsigned int hist[BINS];
    unsigned long long sum = 0;
    double t[REPS];
    for (int r = -1; r < REPS; ++r) {
        double t0 = now_s();
        histogram_cpu(data, n, hist);
        sum = sum_cpu(data, n);
        double dt = now_s() - t0;
        if (r >= 0)
            t[r] = dt;
    }
    qsort(t, REPS, sizeof(double), cmp_double);

    unsigned long long checksum = 0, total = 0;
    int bin_max = 0;
    for (int b = 0; b < BINS; ++b) {
        checksum += (unsigned long long)(b + 1) * hist[b];
        total += hist[b];
        if (hist[b] > hist[bin_max])
            bin_max = b;
    }
    printf("c_histograma_cpu: n=%ld bins=%d t_cpu_ms=%.3f (mediana de %d; min %.3f, max %.3f)\n",
           n, BINS, t[REPS / 2] * 1e3, REPS, t[0] * 1e3, t[REPS - 1] * 1e3);
    printf("  suma=%llu hist_checksum=%llu bin_max=%d cuenta_max=%u total_cuentas=%llu (%s n)\n",
           sum, checksum, bin_max, hist[bin_max], total, total == (unsigned long long)n ? "==" : "!=");
    printf("  hist[0..7]=%u %u %u %u %u %u %u %u\n", hist[0], hist[1], hist[2], hist[3], hist[4], hist[5],
           hist[6], hist[7]);
    if (dump) {
        FILE *f = fopen(dump, "w");
        if (!f) {
            fprintf(stderr, "No se pudo escribir %s\n", dump);
            return 1;
        }
        for (int b = 0; b < BINS; ++b)
            fprintf(f, "%d,%u\n", b, hist[b]);
        fclose(f);
        printf("  256 cuentas escritas en %s\n", dump);
    }
    free(data);
    return total == (unsigned long long)n ? 0 : 2;
}

In [ ]:
%%writefile c_jacobi_cpu.c
/*
 * c_jacobi_cpu.c — Referencia secuencial de la Parte C, opción 2: stencil 2D de Jacobi con doble
 * búfer y criterio de convergencia por reducción máxima.
 *
 * Uso:      ./c_jacobi_cpu [alto=512] [ancho=512] [tol=1e-3] [max_iter=500] [check_every=10]
 * Compilar: gcc -std=c11 -O2 -Wall -Wextra c_jacobi_cpu.c -o c_jacobi_cpu
 *
 * ENTRADA (la misma que debe usar tu kernel; el generador está también en common.cuh):
 *   malla alto×ancho de float aplanada por filas (idx = fila*ancho + col); fila 0 = 100, resto 0.
 * ALGORITMO (tu GPU debe hacer exactamente esto para que los números coincidan):
 *   repetir: out[r][c] = 0.25*(in[r-1][c] + in[r+1][c] + in[r][c-1] + in[r][c+1]) en el interior;
 *            la frontera se copia sin cambio; se intercambian los punteros (doble búfer).
 *            Cada check_every iteraciones se calcula max_diff = máx |out - in| (reducción máxima) y
 *            se detiene si max_diff < tol. También se detiene al llegar a max_iter.
 * SALIDA que tu programa debe reproducir: iteraciones (exacto), motivo de paro, max_diff final y
 *   checksum = Σ malla final (con tolerancia relativa 1e-4: la GPU redondea distinto).
 * Con mallas grandes Jacobi converge MUY despacio (el error cae como (1 − π²/2N²)^k): con 512×512 y
 * tol = 1e-3 manda max_iter. Es normal; repórtalo. Con 64×64 y tol = 1e-2 sí converge antes.
 * T(1): 1 calentamiento + 5 repeticiones de la ejecución completa, mediana (1 si alto·ancho·max_iter > 3e8).
 */
#define _POSIX_C_SOURCE 200809L
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <math.h>
#include <time.h>

#define REPS 5

static double now_s(void)
{
    struct timespec ts;
    clock_gettime(CLOCK_MONOTONIC, &ts);
    return (double)ts.tv_sec + (double)ts.tv_nsec * 1e-9;
}

/* Idéntico a init_jacobi_grid() de common.cuh. */
static void init_grid(float *grid, int height, int width)
{
    for (int r = 0; r < height; ++r)
        for (int c = 0; c < width; ++c)
            grid[(size_t)r * width + c] = (r == 0) ? 100.0f : 0.0f;
}

/* Un paso de Jacobi: interior desde in hacia out; la frontera se copia. */
static void jacobi_step(const float *in, float *out, int height, int width)
{
    for (int r = 0; r < height; ++r)
        for (int c = 0; c < width; ++c) {
            size_t idx = (size_t)r * width + c;
            if (r == 0 || r == height - 1 || c == 0 || c == width - 1)
                out[idx] = in[idx];
            else
                out[idx] = 0.25f * (in[idx - width] + in[idx + width] + in[idx - 1] + in[idx + 1]);
        }
}

static float max_abs_diff(const float *a, const float *b, size_t count)
{
    float m = 0.0f;
    for (size_t i = 0; i < count; ++i) {
        float d = fabsf(a[i] - b[i]);
        if (d > m)
            m = d;
    }
    return m;
}

/* Ejecuta Jacobi completo. Regresa iteraciones; deja la malla final en *final y max_diff en *last_diff. */
static int jacobi_run(float *a, float *b, int height, int width, float tol, int max_iter, int check_every,
                      float **final, float *last_diff, int *converged)
{
    size_t count = (size_t)height * width;
    init_grid(a, height, width);
    memcpy(b, a, count * sizeof(float)); /* la frontera de b también debe estar bien */
    float *in = a, *out = b;
    int iter = 0;
    *last_diff = -1.0f;
    *converged = 0;
    while (iter < max_iter) {
        jacobi_step(in, out, height, width);
        ++iter;
        if (iter % check_every == 0) {
            *last_diff = max_abs_diff(out, in, count); /* reducción máxima */
            if (*last_diff < tol) {
                *converged = 1;
                float *tmp = in; in = out; out = tmp;
                break;
            }
        }
        float *tmp = in; in = out; out = tmp; /* doble búfer: intercambio de punteros */
    }
    *final = in;
    return iter;
}

static int cmp_double(const void *x, const void *y)
{
    double dx = *(const double *)x, dy = *(const double *)y;
    return (dx > dy) - (dx < dy);
}

int main(int argc, char **argv)
{
    int height = (argc > 1) ? atoi(argv[1]) : 512;
    int width = (argc > 2) ? atoi(argv[2]) : 512;
    float tol = (argc > 3) ? (float)atof(argv[3]) : 1e-3f;
    int max_iter = (argc > 4) ? atoi(argv[4]) : 500;
    int check_every = (argc > 5) ? atoi(argv[5]) : 10;
    if (height < 3 || width < 3 || tol <= 0 || max_iter <= 0 || check_every <= 0) {
        fprintf(stderr, "uso: %s [alto>=3] [ancho>=3] [tol>0] [max_iter>0] [check_every>0]\n", argv[0]);
        return 1;
    }
    size_t count = (size_t)height * width;
    float *a = malloc(count * sizeof(float)), *b = malloc(count * sizeof(float));
    if (!a || !b) {
        fprintf(stderr, "Sin memoria para %dx%d\n", height, width);
        return 1;
    }
    int reps = ((double)count * max_iter <= 3e8) ? REPS : 1;
    double t[REPS];
    float *final = NULL, last_diff = 0.0f;
    int iters = 0, converged = 0;
    for (int r = -1; r < reps; ++r) {
        double t0 = now_s();
        iters = jacobi_run(a, b, height, width, tol, max_iter, check_every, &final, &last_diff, &converged);
        double dt = now_s() - t0;
        if (r >= 0)
            t[r] = dt;
    }
    qsort(t, (size_t)reps, sizeof(double), cmp_double);

    double checksum = 0.0;
    for (size_t i = 0; i < count; ++i)
        checksum += final[i];

    printf("c_jacobi_cpu: alto=%d ancho=%d tol=%g max_iter=%d check_every=%d t_cpu_ms=%.3f (mediana de %d; "
           "min %.3f, max %.3f)\n",
           height, width, tol, max_iter, check_every, t[reps / 2] * 1e3, reps, t[0] * 1e3, t[reps - 1] * 1e3);
    printf("  iteraciones=%d paro=%s max_diff=%.6e checksum=%.6f centro=%.6f\n", iters,
           converged ? "convergio" : "max_iter", last_diff, checksum, final[(size_t)(height / 2) * width + width / 2]);
    free(a);
    free(b);
    return 0;
}

In [ ]:
%%writefile graficar.py
#!/usr/bin/env python3
"""graficar.py — Gráficas y tabla de la Parte D a partir del CSV del curso (esquema de common.cuh / S04).

Uso:
    python3 graficar.py resultados/resultados_P6.csv [otro.csv ...] [-o carpeta_salida]

Lee uno o varios CSV (se concatenan), y escribe en la carpeta de salida (por defecto la del primer CSV):
    tiempo_vs_bloque.png   tiempo del kernel vs. hilos por bloque (filas con experimento = "bloque"), un panel
                           por kernel; marca el mejor bloque y muestra el total con copias en línea discontinua
    speedup_vs_n.png       speedup vs. n (filas "umbral"; si el CSV no trae ninguna, un kernel con >= 3 tamaños al
                           mismo bloque):
                           solo kernel (continua) y con copias (discontinua); la línea S = 1 es el empate con la CPU.
                           Si el CSV trae la columna opcional t_omp4_ms (V2 OpenMP con 4 hilos), agrega esa curva
    transferencias.png     para el n más grande de cada kernel: qué porcentaje del total es H2D / kernel / D2H, y
                           el total en ms al final de cada barra
Imprime además la tabla en Markdown (para pegar en REPORTE.md) y un resumen: mejor bloque por kernel, primer n
con speedup > 1 sin y con copias, y porcentaje de copias. El título dice si los datos son medidos (origen=gpu,
con el modelo de GPU) o ilustrativos/de prueba. Solo usa la biblioteca estándar + matplotlib.
"""
import argparse
import csv
import os
import sys

import matplotlib

matplotlib.use("Agg")  # siempre a archivo: funciona sin pantalla y dentro de Colab
import matplotlib.pyplot as plt  # noqa: E402

# Paleta categórica validada (orden fijo, nunca cíclico) + marcador distinto por serie (codificación secundaria).
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#6250d6", "#008300", "#e34948"]
MARKERS = ["o", "s", "^", "D", "v", "P", "X", "*"]
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e5e4df"
NUMERIC = ("t_h2d_ms", "t_kernel_ms", "t_d2h_ms", "t_total_ms", "t_cpu_ms", "speedup_kernel", "speedup_total",
           "ancho_banda_gbs", "ocupacion_pct")


def read_rows(paths):
    rows = []
    for path in paths:
        with open(path, newline="", encoding="utf-8") as f:
            for r in csv.DictReader(f):
                try:
                    for key in ("n", "block", "grid"):
                        r[key] = int(float(r[key]))
                    for key in NUMERIC:
                        r[key] = float(r[key])
                    omp = r.get("t_omp4_ms", "")
                    r["t_omp4_ms"] = float(omp) if omp not in ("", None) else None
                except (KeyError, ValueError) as exc:
                    sys.exit(f"{path}: fila inválida ({exc}): {r}")
                rows.append(r)
    return rows


def style_map(rows):
    """Color y marcador por kernel, en orden de primera aparición: el color sigue a la entidad, no a su rango.
    Con más de 8 kernels (Partes A, B y C juntas) los colores se repiten solo a partir del noveno; en
    tiempo_vs_bloque.png cada kernel tiene su propio panel y en speedup_vs_n.png solo entran los del barrido "umbral",
    así que nunca se confunden dos series en la misma gráfica. Ningún kernel se omite."""
    kernels = []
    for r in rows:
        if r["kernel"] not in kernels:
            kernels.append(r["kernel"])
    return {k: (PALETTE[i % len(PALETTE)], MARKERS[i % len(MARKERS)]) for i, k in enumerate(kernels)}


def source_note(rows):
    origins = {r["origen"] for r in rows}
    if origins == {"gpu"}:
        return "medido en " + ", ".join(sorted({r["dispositivo"] for r in rows}))
    return "DATOS NO MEDIDOS (origen: " + ", ".join(sorted(origins)) + ")"


def tidy(ax):
    ax.grid(axis="y", color=GRID, linewidth=0.8)
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(colors=MUTED)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(GRID)


def fmt_n(n):
    return f"{n:,}".replace(",", " ")


def plot_block_sweep(rows, out_path, styles, note):
    rows = [r for r in rows if r["experimento"] == "bloque"]
    if not rows:
        print("Sin filas con experimento=bloque: no se genera tiempo_vs_bloque.png")
        return
    kernels = [k for k in styles if any(r["kernel"] == k for r in rows)]
    ncols = min(3, len(kernels))  # paneles en filas de 3: legibles en el reporte y en pantalla
    nrows = (len(kernels) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.6 * ncols, 3.9 * nrows + 0.4), squeeze=False)
    flat = [ax for row_axes in axes for ax in row_axes]
    for ax in flat[len(kernels):]:
        ax.set_visible(False)  # huecos de la última fila
    for ax, kernel in zip(flat, kernels):
        sub = [r for r in rows if r["kernel"] == kernel]
        n_big = max(r["n"] for r in sub)  # si hay varios n, se grafica el mayor (el del protocolo)
        pts = sorted((r["block"], r["t_kernel_ms"], r["t_total_ms"]) for r in sub if r["n"] == n_big)
        color, marker = styles[kernel]
        xs = list(range(len(pts)))
        ax.plot(xs, [p[1] for p in pts], "-", marker=marker, markersize=7, linewidth=2, color=color,
                label="solo kernel")
        ax.plot(xs, [p[2] for p in pts], "--", marker=marker, markersize=5, linewidth=1.5, color=color,
                alpha=0.75, label="con copias (total)")
        best = min(pts, key=lambda p: p[1])
        ax.annotate(f"mejor: {best[0]} hilos\n{best[1]:.3f} ms", (xs[pts.index(best)], best[1]),
                    textcoords="offset points", xytext=(0, 12), ha="center", fontsize=8, color=MUTED)
        ax.set_xticks(xs)
        ax.set_xticklabels([str(p[0]) for p in pts])
        ax.set_xlabel("hilos por bloque")
        ax.set_ylabel("tiempo (ms)")
        ax.set_yscale("log")
        ax.set_title(f"{kernel} · n = {fmt_n(n_big)}", fontsize=10, color=INK)
        ax.legend(frameon=False, fontsize=8)
        tidy(ax)
    fig.suptitle(f"Tiempo vs. tamaño de bloque · {note}", fontsize=11, color=INK)
    fig.tight_layout()
    fig.savefig(out_path, dpi=130)
    plt.close(fig)
    print(f"Gráfica escrita: {out_path}")


def threshold_rows(rows, kernel):
    """Filas 'umbral' del kernel. Solo si el CSV no trae ningún barrido 'umbral' (p. ej. un CSV con tu Parte C sola),
    se usan las del bloque más frecuente cuando hay >= 3 tamaños distintos: así no se mezclan en el mismo eje un n que
    es el lado de una matriz con un n que es número de elementos. Las filas 'uno' (configuraciones sueltas, como las
    formas rectangulares de la Parte B) nunca forman un barrido."""
    sub = [r for r in rows if r["kernel"] == kernel and r["experimento"] == "umbral"]
    if not sub and not any(r["experimento"] == "umbral" for r in rows):
        cand = [r for r in rows if r["kernel"] == kernel and r["experimento"] != "uno"]
        if not cand:
            return []
        blocks = {}
        for r in cand:
            blocks[r["block"]] = blocks.get(r["block"], 0) + 1
        block = max(blocks, key=blocks.get)
        sub = [r for r in cand if r["block"] == block]
        if len({r["n"] for r in sub}) < 3:
            return []
    sub.sort(key=lambda r: r["n"])
    return sub


def plot_threshold(rows, out_path, styles, note):
    series = {k: threshold_rows(rows, k) for k in styles}
    series = {k: v for k, v in series.items() if v}
    if not series:
        print("Sin kernels con >= 3 tamaños de n: no se genera speedup_vs_n.png")
        return
    fig, ax = plt.subplots(figsize=(7.4, 4.8))
    for kernel, sub in series.items():
        color, marker = styles[kernel]
        ns = [r["n"] for r in sub]
        ax.plot(ns, [r["speedup_kernel"] for r in sub], "-", marker=marker, markersize=7, linewidth=2,
                color=color, label=f"{kernel}: solo kernel")
        ax.plot(ns, [r["speedup_total"] for r in sub], "--", marker=marker, markersize=5, linewidth=1.5,
                color=color, alpha=0.8, label=f"{kernel}: con copias")
        if all(r["t_omp4_ms"] is not None for r in sub):
            ax.plot(ns, [r["t_omp4_ms"] / r["t_total_ms"] for r in sub], ":", marker=marker, markersize=5,
                    linewidth=1.5, color=color, alpha=0.8, label=f"{kernel}: con copias vs. OpenMP 4 hilos")
    ax.axhline(1.0, color=MUTED, linewidth=1)
    ax.text(min(min(r["n"] for r in s) for s in series.values()), 1.0, " S = 1 (empate con la CPU)",
            va="bottom", fontsize=8, color=MUTED)
    ax.set_xscale("log", base=2)
    ax.set_yscale("log")
    ax.set_xlabel("n (elementos)")
    ax.set_ylabel("speedup = T_cpu / T_gpu")
    ax.set_title(f"¿A partir de qué n gana la GPU? · {note}", fontsize=11, color=INK)
    ax.legend(frameon=False, fontsize=8, ncol=2)
    tidy(ax)
    ax.grid(axis="x", color=GRID, linewidth=0.8)
    fig.tight_layout()
    fig.savefig(out_path, dpi=130)
    plt.close(fig)
    print(f"Gráfica escrita: {out_path}")


def largest_rows(rows, styles):
    out = []
    for kernel in styles:
        sub = [r for r in rows if r["kernel"] == kernel]
        if sub:
            n_big = max(r["n"] for r in sub)
            out.append(min((r for r in sub if r["n"] == n_big), key=lambda r: r["t_kernel_ms"]))
    return out


def plot_transfers(rows, out_path, styles, note):
    """Una barra por kernel (su n mayor): qué fracción del tiempo total es H2D, kernel y D2H. Se grafican
    proporciones porque los totales de distintos kernels difieren en órdenes de magnitud; el total va en ms al final."""
    rows = largest_rows(rows, styles)
    if not rows:
        return
    fig, ax = plt.subplots(figsize=(7.8, 1.2 + 0.62 * len(rows)))
    labels = [f"{r['kernel']}\nn = {fmt_n(r['n'])}, bloque {r['block']}" for r in rows]
    parts = (("t_h2d_ms", "H2D", "#8a8984"), ("t_kernel_ms", "kernel", "#2a78d6"), ("t_d2h_ms", "D2H", "#c3c2b7"))
    left = [0.0] * len(rows)
    for key, name, color in parts:
        pcts = [100.0 * r[key] / r["t_total_ms"] for r in rows]
        ax.barh(labels, pcts, left=left, color=color, label=name, height=0.6, edgecolor="white", linewidth=1.5)
        for i, pct in enumerate(pcts):
            if pct >= 12:  # solo se rotula el tramo que tiene espacio; el resto se lee en la tabla impresa
                ax.text(left[i] + pct / 2, i, f"{pct:.0f} %", ha="center", va="center", fontsize=8,
                        color="white" if key == "t_kernel_ms" else INK)
        left = [a + b for a, b in zip(left, pcts)]
    for i, r in enumerate(rows):
        ax.text(101.5, i, f"{r['t_total_ms']:.2f} ms", va="center", fontsize=8, color=MUTED)
    ax.set_xlim(0, 118)
    ax.set_xticks([0, 25, 50, 75, 100])
    ax.set_xlabel("% del tiempo total con copias (mediana de 5); a la derecha, el total en ms")
    ax.set_title(f"¿Cuánto pesan las copias? · {note}", fontsize=11, color=INK)
    ax.legend(frameon=False, fontsize=8, ncol=3, loc="upper center", bbox_to_anchor=(0.43, -0.12 - 0.3 / len(rows)))
    ax.invert_yaxis()
    tidy(ax)
    ax.grid(axis="y", visible=False)
    ax.grid(axis="x", color=GRID, linewidth=0.8)
    fig.tight_layout()
    fig.savefig(out_path, dpi=130)
    plt.close(fig)
    print(f"Gráfica escrita: {out_path}")


def first_crossing(sub, key):
    for r in sub:
        if r[key] > 1.0:
            return r["n"]
    return None


def print_summary(rows, styles):
    print("\n## Tabla (ms, mediana de 5 tras 1 calentamiento)\n")
    print("| experimento | kernel | n | bloque | H2D | kernel | D2H | total | CPU | S_kernel | S_total | copias % | verif |")
    print("|---|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---|")
    for r in sorted(rows, key=lambda r: (r["experimento"], r["kernel"], r["n"], r["block"])):
        copies = 100.0 * (r["t_h2d_ms"] + r["t_d2h_ms"]) / r["t_total_ms"]
        verif = r.get("verificacion", "")  # columna opcional
        print(f"| {r['experimento']} | {r['kernel']} | {r['n']} | {r['block']} | {r['t_h2d_ms']:.4f} | "
              f"{r['t_kernel_ms']:.4f} | {r['t_d2h_ms']:.4f} | {r['t_total_ms']:.4f} | {r['t_cpu_ms']:.4f} | "
              f"{r['speedup_kernel']:.2f} | {r['speedup_total']:.2f} | {copies:.0f} | {verif} |")
    print("\n## Resumen\n")
    block_rows = [r for r in rows if r["experimento"] == "bloque"]
    for kernel in styles:
        sub = [r for r in block_rows if r["kernel"] == kernel]
        if sub:
            n_big = max(r["n"] for r in sub)
            sub = [r for r in sub if r["n"] == n_big]
            best, worst = min(sub, key=lambda r: r["t_kernel_ms"]), max(sub, key=lambda r: r["t_kernel_ms"])
            print(f"- {kernel} (n={n_big}): mejor bloque {best['block']} ({best['t_kernel_ms']:.4f} ms), peor "
                  f"{worst['block']} ({worst['t_kernel_ms']:.4f} ms), peor/mejor = "
                  f"{worst['t_kernel_ms'] / best['t_kernel_ms']:.2f}x; ocupación del mejor {best['ocupacion_pct']:.0f} %")
    for kernel in styles:
        sub = threshold_rows(rows, kernel)
        if sub:
            fmt = lambda v: f"desde n >= {v}" if v else "no cruza en el rango medido"
            print(f"- {kernel}: solo kernel, la GPU gana {fmt(first_crossing(sub, 'speedup_kernel'))}; "
                  f"con copias, {fmt(first_crossing(sub, 'speedup_total'))}")
    for r in largest_rows(rows, styles):
        copies = 100.0 * (r["t_h2d_ms"] + r["t_d2h_ms"]) / r["t_total_ms"]
        print(f"- {r['kernel']} (n={r['n']}, bloque {r['block']}): copias = {copies:.0f} % del total; "
              f"{r['ancho_banda_gbs']:.1f} GB/s útiles en el kernel")


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("csv", nargs="+", help="uno o más CSV con el esquema del curso")
    ap.add_argument("-o", "--salida", help="carpeta de salida (por defecto, la del primer CSV)")
    args = ap.parse_args()
    out_dir = args.salida or (os.path.dirname(args.csv[0]) or ".")
    os.makedirs(out_dir, exist_ok=True)
    rows = read_rows(args.csv)
    if not rows:
        sys.exit("El CSV no tiene filas")
    styles = style_map(rows)
    note = source_note(rows)
    print(f"{len(rows)} filas leídas · {note}")
    plot_block_sweep(rows, os.path.join(out_dir, "tiempo_vs_bloque.png"), styles, note)
    plot_threshold(rows, os.path.join(out_dir, "speedup_vs_n.png"), styles, note)
    plot_transfers(rows, os.path.join(out_dir, "transferencias.png"), styles, note)
    print_summary(rows, styles)


if __name__ == "__main__":
    main()

## 2 · Compilar

`nvcc` compila el código de anfitrión con el compilador de C++ del sistema y el de dispositivo para la GPU. En una T4 cada línea equivale a `nvcc -O2 -arch=sm_75 archivo.cu -o archivo`. La multiplicación por mosaicos de la Parte A se compila con tres tamaños de mosaico (`-DTILE=8|16|32`).

In [ ]:
# En una T4 (Colab) esto equivale a:  !nvcc -O2 -arch=sm_75 a_vectores.cu -o a_vectores
!nvcc -O2 -arch={ARCH} a_vectores.cu -o a_vectores
!for t in 8 16 32; do nvcc -O2 -arch={ARCH} -DTILE=$t a_matmul_tiled.cu -o a_matmul_tiled_t$t; done
!nvcc -O2 -arch={ARCH} b_matmul_rect.cu -o b_matmul_rect
!gcc -std=c11 -O2 -Wall -Wextra a_vectores_cpu.c -o a_vectores_cpu -lm
!gcc -std=c11 -O2 -Wall -Wextra a_matmul_cpu.c -o a_matmul_cpu
!gcc -std=c11 -O2 -Wall -Wextra b_matmul_rect_cpu.c -o b_matmul_rect_cpu
!gcc -std=c11 -O2 -Wall -Wextra c_histograma_cpu.c -o c_histograma_cpu
!gcc -std=c11 -O2 -Wall -Wextra c_jacobi_cpu.c -o c_jacobi_cpu -lm
!ls -l a_vectores a_matmul_tiled_t* b_matmul_rect *_cpu

## 3 · Parte A.1–A.3 · Suma de vectores y SAXPY

Primero `n = 1000`, el caso de tus predicciones (A.1): compara los *checksums* de la CPU, la GPU y el espejo Numba. Después el tamaño por defecto, `n = 2^24` (64 MB por vector).

In [ ]:
!./a_vectores_cpu 1000
!./a_vectores uno 1000 256
!python3 a_vectores_numba.py 1000 256

In [ ]:
!./a_vectores_cpu 16777216
!./a_vectores uno 16777216 256
# Espejo Numba en la GPU real (la primera llamada compila el kernel: por eso hay calentamiento)
!python3 a_vectores_numba.py 16777216 256

### Tabla de la Parte A (A.3): tres tamaños, con y sin copias

Una configuración por tamaño con bloque 256. Copia a la tabla del README, para cada kernel, `kernel ms`, `total ms` y `CPU ms`; la columna `S_total` es el *speedup* **con** copias y `S_kern` **sin** ellas. Anota también qué porcentaje del total son las copias (H2D + D2H).

In [ ]:
%%bash
for n in 100000 1000000 10000000; do ./a_vectores uno $n 256; done

## 4 · Parte A.4 · Medir: umbral de tamaño y barrido de bloque

Todo se acumula en `resultados_P6.csv` con el esquema del curso. `umbral` recorre `n = 2^10 … 2^26` con bloque 256; `bloque` recorre 32 … 1024 con `n = 2^24`. Tarda 1–2 minutos en una T4.

In [ ]:
import os
if os.path.exists("resultados_P6.csv"):
    os.remove("resultados_P6.csv")
!./a_vectores umbral 256 resultados_P6.csv
!./a_vectores bloque 16777216 resultados_P6.csv

## 5 · Parte A.6–A.8 · Multiplicación de matrices: ingenua vs. por mosaicos (código completo)

Es el *kernel* que rastreaste en la S02. **Antes de ejecutar llena la tabla A.7 del README** (lecturas, fases, bloques, hilos que cargan ceros). Criterio: `OK` en las dos versiones con `n = 512`, `1024` **y** `1000` (no múltiplo de 16). `a_matmul_cpu` imprime el modelo de lecturas a memoria global.

In [ ]:
!./a_matmul_cpu 1000 16
!./a_matmul_tiled_t16 512 resultados_P6.csv
!./a_matmul_tiled_t16 1000 resultados_P6.csv
!./a_matmul_tiled_t16 1024 resultados_P6.csv

In [ ]:
# Espejo Numba en la GPU real: mismo kernel; compara su checksum con el de CUDA C
!python3 a_matmul_numba.py 512 16

In [ ]:
%%bash
# Barrido TILE x n para la Parte D (cada TILE es un tamaño de bloque: 64, 256 y 1024 hilos). ~1 min en T4.
for n in 512 1024 2048; do
  for t in 8 16 32; do ./a_matmul_tiled_t$t $n resultados_P6.csv | grep -E "matmul_(naive|tiled) "; done
done

## 6 · Parte B · Mosaicos con matrices rectangulares (`TODO(1)`–`TODO(4)`)

La ingenua rectangular y la CPU corren siempre. La de mosaicos imprime `TODO pendiente` hasta que corrijas `TODO(1)`–`TODO(4)` en la celda `%%writefile b_matmul_rect.cu` (y pongas `TODO_LISTO 1`), la vuelvas a ejecutar y recompiles (sección 2). Verifica primero la lógica con el simulador (sirve también aquí, en Colab): las dos pruebas de `make test-b`.

In [ ]:
%%bash
# Lógica sin GPU (simulador): la cuadrada pasa aun sin corregir; la rectangular es la que decide.
NUMBA_ENABLE_CUDASIM=1 python3 b_matmul_rect_numba.py 32 32 32 16 tiled 2>/dev/null
NUMBA_ENABLE_CUDASIM=1 python3 b_matmul_rect_numba.py 20 40 12 16 tiled 2>/dev/null

In [ ]:
%%bash
# En la GPU: tres formas (una cuadrada, dos rectangulares con dimensiones no múltiplos de 16).
./b_matmul_rect_cpu 1000 600 700 16
for dims in "512 512 512" "1000 600 700" "300 2000 900"; do ./b_matmul_rect $dims resultados_P6.csv; done

## 7 · Parte C · Diseño propio

Las referencias CPU dan la salida que tu *kernel* debe reproducir (histograma/reducción: exacto; Jacobi: iteraciones exactas y malla con tolerancia). Tu `src/c_<nombre>.cu` (y su espejo `src/c_<nombre>_numba.py`, si lo hiciste) entra solo a la sección 1 cuando corres `make notebook`; compílalo aquí y haz que agregue sus filas a `resultados_P6.csv` con `emit_row` de `common.cuh` (experimento `bloque`, una fila por tamaño de bloque).

In [ ]:
!./c_histograma_cpu 16777216 --volcar hist_cpu.txt
!./c_jacobi_cpu 1024 1024 1e-3 500 10
!./c_jacobi_cpu 64 64 1e-2 1000 10

In [ ]:
# Plantilla para tu Parte C (descomenta y completa; c_mi_kernel.cu ya se escribió en la sección 1 con make notebook):
# !nvcc -O2 -arch={ARCH} c_mi_kernel.cu -o c_mi_kernel
# !for b in 32 64 128 256 512 1024; do ./c_mi_kernel <args> $b resultados_P6.csv; done

## 8 · Parte D · Gráficas e interpretación

`graficar.py` lee el CSV y produce `tiempo_vs_bloque.png`, `speedup_vs_n.png` y `transferencias.png`, más la tabla en Markdown para `REPORTE.md`. Las tres preguntas de interpretación están en el README (D.3).

In [ ]:
!python3 graficar.py resultados_P6.csv -o resultados
from IPython.display import Image, display
for png in ("tiempo_vs_bloque.png", "speedup_vs_n.png", "transferencias.png"):
    display(Image(filename=f"resultados/{png}"))

## 9 · Guardar resultados (CSV, PNG y entorno)

Se agrega la fecha, se guarda `entorno.txt` con la GPU y la CPU de esta sesión y se descarga todo. Súbelo a `practicas/P6/resultados/` del repositorio de tu equipo. En Kaggle los archivos quedan en `/kaggle/working/` (panel *Output*).

In [ ]:
import datetime, os, shutil, subprocess
import pandas as pd
os.makedirs("resultados", exist_ok=True)  # por si no corriste la sección 8
hoy = datetime.date.today().isoformat()
df = pd.read_csv("resultados_P6.csv")
df["fecha"] = hoy
df.to_csv(f"resultados/resultados_P6_{hoy}.csv", index=False)
with open("resultados/entorno.txt", "w") as f:
    f.write(f"fecha: {hoy}\n")
    f.write(subprocess.run("nvidia-smi --query-gpu=name,driver_version,memory.total,compute_cap --format=csv", shell=True, capture_output=True, text=True).stdout)
    f.write(subprocess.run("nvcc --version | tail -1; lscpu | grep -E 'Model name|^CPU\\(s\\)'", shell=True, capture_output=True, text=True).stdout)
print(open("resultados/entorno.txt").read())
print(df.groupby(["experimento", "kernel"]).size())
shutil.make_archive(f"resultados_P6_{hoy}", "zip", "resultados")
try:
    from google.colab import files  # solo existe en Colab
    files.download(f"resultados_P6_{hoy}.zip")
except ImportError:
    print("En Kaggle: descarga resultados_P6_*.zip desde el panel Output (/kaggle/working/).")

## Qué te llevas

1. `resultados_P6_<fecha>.csv`, las tres PNG y `entorno.txt` (GPU, CPU, fecha): van en `resultados/` del repositorio.
2. La tabla Markdown que imprimió `graficar.py`: va en `REPORTE.md` con tu interpretación (D.3) y tu pre-mortem comparado.
3. La bitácora: cada `FALLA`, `TODO pendiente` o `invalid configuration argument` que viste aquí es una entrada.